# StratScout — Prototype (single-file)

Tactical recognition for CS2 from **minimap pixels only**.

```
demo replay .mp4
  -> auto-crop the radar from the top-left of the screen
  -> Model 1  (CNN)  detect orange T / blue CT icons
  -> tracker         link icons across frames into 10 stable slots
  -> trajectories    (32, 10, 2) -> feature sequence
  -> Model 2  (RNN)  classify the round's tactic
  -> scouting output heatmaps, tendencies, per-round predictions
```

**The one rule:** at inference the system sees pixels and nothing else. `.dem` files may be
used during development as ground truth, never as an input to the deployed pipeline.

**Two questions per round.**

1. *Before the plant, what did the attackers do?* `A_RUSH`, `B_RUSH`, `A_SPLIT`, `B_SPLIT`, `DEFAULT` (slow map control).
2. *After the plant, how did the defenders respond?* `RETAKE` (together), `TRICKLE` (one at a time), `SAVE` (gave the site up). Asked only when the bomb was planted; the site comes from where it was planted.

**This notebook runs with or without your footage.** Drop rendered demo replays (`.mp4`) into
the `data/` folder and it uses real frames; leave the folder empty and every stage falls back to
a synthetic generator so the whole pipeline still executes end to end. Cells say which mode they
are in.

**One `.mp4` in `data/` = one match.** The file name (without `.mp4`) becomes the `match_id`, and
train/val/test splits are made by match — so never put two matches in one file, and never cut one
match into several files.

> Numbers produced in synthetic mode are **not results**. The generator draws the dots and the
> detector then learns to find dots the generator drew — a closed loop. It proves the wiring,
> not the idea.

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import cv2
except ImportError:
    cv2 = None
    print("opencv not installed — video cells will be skipped.  pip install opencv-python")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)


@dataclass
class Config:
    seed: int = 0

    # --- input video -------------------------------------------------------
    data_dir: str = "data"                # rendered demo replays live here, one .mp4 per match
    video_exts: Tuple[str, ...] = (".mp4",)   # matched case-insensitively (.MP4 works too)
    video_path: Optional[str] = None      # None -> first video in data_dir; set to pin one
    crop_box: Optional[Tuple[int, int, int, int]] = None   # None -> detected automatically
    probe_frames: int = 20                # frames sampled to locate the radar
    search_frac: float = 0.45             # radar is searched for in this fraction of the screen
    start_s: float = 60.0                 # skip warmup / map intro
    end_s: Optional[float] = None         # None -> a few seconds before the end

    # --- minimap -----------------------------------------------------------
    minimap_px: int = 128
    n_per_team: int = 5
    teams: Tuple[str, str] = ("T", "CT")  # T = orange (attackers), CT = blue (defenders)

    # --- sequences ---------------------------------------------------------
    seq_len: int = 32
    fps_sampled: float = 4.0

    # --- classical icon finder --------------------------------------------
    sat_min: int = 90
    val_min: int = 90
    icon_area: Tuple[int, int] = (6, 250)
    hue_tol: float = 22.0
    bg_thresh: int = 40

    # --- detector (Model 1) ------------------------------------------------
    det_stride: int = 1
    det_sigma: float = 1.0
    det_epochs: int = 25
    det_lr: float = 2e-3
    det_batch: int = 32
    det_peak_thresh: float = 0.20
    det_match_radius: float = 0.035

    # --- tracker -----------------------------------------------------------
    track_gate: float = 0.09
    track_max_miss: int = 8

    # --- classifier (Model 2) ---------------------------------------------
    cls_epochs: int = 40
    cls_lr: float = 2e-3
    cls_batch: int = 32
    cls_hidden: int = 96

    # --- files -------------------------------------------------------------
    map_image: str = "mirage_radar.png"   # clean radar screenshot: background for every plot
    frames_npy: str = "pilot_frames.npy"
    icon_labels_json: str = "pilot_labels.json"
    frames_src_json: str = "pilot_frames_sources.json"   # which video each pilot frame came from
    crops_json: str = "crops.json"        # cached crop box per video (auto-crop is slow on big files)
    labels_dir: str = "labels"            # round labels, one <match_id>.rounds.json per video
    rounds_npz: str = "rounds_data.npz"
    out_dir: str = "runs/prototype"


CFG = Config()

# Two questions per round:
#   1. before the plant, what did the attackers do?    -> T_CLASSES, every round
#   2. after the plant, how did the defenders respond?  -> CT_CLASSES, planted rounds only
# The site comes from where the bomb went down, so neither label needs to name it.
T_CLASSES = ["A_RUSH", "B_RUSH", "A_SPLIT", "B_SPLIT", "DEFAULT"]
CT_CLASSES = ["RETAKE", "TRICKLE", "SAVE"]
PHASES = {"pre": T_CLASSES, "post": CT_CLASSES}     # phase of the round -> labels read in it
CLASSES = T_CLASSES                                 # the headline (attacker) question
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)
N_PLAYERS = CFG.n_per_team * 2


def set_seed(s: int) -> None:
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(CFG.seed)
Path(CFG.out_dir).mkdir(parents=True, exist_ok=True)

def list_videos(data_dir: Optional[str] = None) -> List[Path]:
    """Every readable replay in the data folder, sorted so the order is stable between runs.

    Drop-in friendly: hidden / temp files are ignored, and a file OpenCV cannot open (still
    copying, truncated render, wrong codec) is skipped with a warning instead of crashing later.
    """
    d = Path(data_dir or CFG.data_dir)
    d.mkdir(parents=True, exist_ok=True)
    found = sorted(p for p in d.iterdir() if p.is_file()
                   and p.suffix.lower() in CFG.video_exts and not p.name.startswith((".", "~")))
    if cv2 is None:
        return found
    ok = []
    for p in found:
        cap = cv2.VideoCapture(str(p))
        n = cap.get(cv2.CAP_PROP_FRAME_COUNT) if cap.isOpened() else 0
        cap.release()
        if n > 0:
            ok.append(p)
        else:
            print(f"WARNING: skipping {p.name} — OpenCV cannot read it (still copying? corrupt?)")
    return ok


def match_id_of(video) -> str:
    """data/faze_vs_navi_m1.mp4 -> 'faze_vs_navi_m1'. One file = one match."""
    return Path(video).stem


VIDEOS = list_videos()
if CFG.video_path is None and VIDEOS:
    CFG.video_path = str(VIDEOS[0])
HAVE_VIDEO = cv2 is not None and CFG.video_path is not None and Path(CFG.video_path).exists()

print(f"data dir: {CFG.data_dir}/  ->  {len(VIDEOS)} video(s)")
for v in VIDEOS:
    print(f"   {v.name:<40} {v.stat().st_size / 1e9:6.2f} GB")
if not VIDEOS:
    print(f"   (empty — drop rendered demo replays into {Path(CFG.data_dir).resolve()})")
print("primary video:", CFG.video_path, "->",
      "FOUND (real mode)" if HAVE_VIDEO else "missing (synthetic mode)")
print("attacker tactics (before the plant):", T_CLASSES)
print("defender responses (after the plant):", CT_CLASSES)

## 2. Data contracts

Five people building five stages, and the thing that breaks the project is not a bad model —
it is one stage returning a list of tuples where the next expected a tensor. So the shapes are
fixed here in writing, and every stage honours them.

Like plug sockets: agree on the plug and you can swap the appliance without rewiring the house.

| Stage | Name | Shape / type | Meaning |
|---|---|---|---|
| ingest | `frames` | `uint8 (N, H, W, 3)` | cropped radar, RGB, 0–255 |
| preprocess | `x` | `float32 (N, 3, 128, 128)` | normalised 0–1, channels first |
| detector | `detections` | list of `{"T": (k,3), "CT": (m,3)}` | rows `(x, y, score)`, normalised |
| tracker | `positions` | `float32 (N, 10, 2)` | slots 0–4 = T, 5–9 = CT |
| trajectory | `sequence` | `float32 (32, D)` | model-ready |
| classifier | `logits` | `float32 (B, 5)` attacker, `(B, 3)` defender | order fixed by `T_CLASSES` / `CT_CLASSES` |

**Coordinates** are normalised to the *crop*, x right, y down. Never pass raw pixels across a
stage boundary — pixels change the moment someone edits the resolution, fractions do not.

**Identity is not promised.** Icons within a team are identical, so when two teammates cross,
slots can swap. The classifier reads team *shape*, not individuals, so this is survivable —
but it is why the tracker keeps fixed slots rather than discovering tracks.

In [ ]:
TEAM_IDX = {"T": 0, "CT": 1}


def check_contract(name: str, arr, expected: Tuple) -> None:
    got = tuple(arr.shape)
    if len(got) != len(expected) or any(e is not None and e != g for e, g in zip(expected, got)):
        raise ValueError(f"[contract] {name}: expected {expected}, got {got}")
    print(f"[contract] {name}: {got} OK")


print("contracts defined")


def show_pipeline():
    """The pipeline as a diagram: each stage, and the data contract it hands to the next."""
    from matplotlib.patches import FancyBboxPatch
    stages = [("Replay", "recorded .mp4"), ("Auto-crop", "find the radar"),
              ("Detector", "CNN, Model 1"), ("Tracker", "10 fixed slots"),
              ("Classifier", "Model 2: two\nquestions"), ("Report", "tactics, heatmaps,\nwin rates")]
    contracts = ["frames\n(N,H,W,3)", "x\n(N,3,128,128)", "detections\n{T, CT}",
                 "positions\n(N,10,2)", "logits\n(B,5) + (B,3)"]
    learned = {"Detector", "Classifier"}
    fig, ax = plt.subplots(figsize=(13, 3.0))
    w, gap = 1.55, 0.62
    for k, (name, sub) in enumerate(stages):
        x = k * (w + gap)
        edge = "#d9822b" if name in learned else "#5b6b7c"
        ax.add_patch(FancyBboxPatch((x, 0), w, 1.0, boxstyle="round,pad=0.02,rounding_size=0.08",
                                    fc="#eef1f4", ec=edge, lw=2.2))
        ax.text(x + w / 2, 0.64, name, ha="center", va="center", fontsize=13, weight="bold")
        ax.text(x + w / 2, 0.30, sub, ha="center", va="center", fontsize=9.5, color="#44505c")
        if k < len(contracts):
            ax.annotate("", xy=(x + w + gap - 0.04, 0.5), xytext=(x + w + 0.04, 0.5),
                        arrowprops=dict(arrowstyle="-|>", color="#33404d", lw=1.6))
            ax.text(x + w + gap / 2, -0.32, contracts[k], ha="center", va="center",
                    fontsize=8.5, family="monospace", color="#33404d")
    ax.set_xlim(-0.1, len(stages) * (w + gap) - gap + 0.1); ax.set_ylim(-0.7, 1.15)
    ax.axis("off")
    ax.set_title("StratScout pipeline: pixels in, tactics out "
                 "(orange outline = learned model; text under arrows = data contract)", fontsize=11)
    plt.tight_layout(); plt.show()


show_pipeline()

## 3. Finding the radar automatically

Previously the crop box was clicked by hand every run. That is both tedious and fragile — four
pixels of slop shifts every coordinate downstream.

**The trick:** a HUD overlay is drawn by the game at exactly the same pixels every frame, while
the rendered 3D world behind it never repeats. So we sample frames from across the whole video
and look in the top-left for a region that is simultaneously

* **perfectly stable** — the pixel is within ±3 of its median in ~every frame, and
* **detailed** — strong edges, which is what rules out flat regions like sky.

Stability alone is not enough: a pale sky barely changes either. Detail alone is not enough:
the world is full of edges. The *intersection* is the radar, and essentially nothing else.

Like spotting a sticker on a train window — the scenery streams past, the sticker never moves,
and unlike the clear glass around it, the sticker has a picture on it.

In [ ]:
def sample_probe_frames(video: str, n: int = 20,
                        skip_head: float = 0.05, skip_tail: float = 0.05) -> np.ndarray:
    """Grab `n` RGB frames spread across the video (skipping intro/outro)."""
    assert cv2 is not None, "pip install opencv-python"
    cap = cv2.VideoCapture(video)
    if not cap.isOpened():
        raise FileNotFoundError(video)
    total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    idx = np.linspace(total * skip_head, total * (1 - skip_tail), n).astype(int)
    out = []
    for i in idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, bgr = cap.read()
        if ok:
            out.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(out) < 4:
        raise RuntimeError("could not read enough frames — is the file a valid video?")
    return np.stack(out)


def video_resolution(video: str) -> Tuple[int, int]:
    cap = cv2.VideoCapture(str(video))
    wh = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return wh


def _stability_and_detail(frames_rgb: np.ndarray):
    g = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in frames_rgb]).astype(np.float32)
    med = np.median(g, axis=0)
    stable = (np.abs(g - med[None]) <= 3.0).mean(axis=0)     # 1.0 = never changes
    gx = cv2.Sobel(med.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(med.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    detail = cv2.blur(np.sqrt(gx * gx + gy * gy), (21, 21))
    return stable, detail, med


def find_radar_box(frames: np.ndarray, search_frac: float = 0.45, stable_min: float = 0.97,
                   detail_min: float = 6.0, pad: int = 2, square: bool = True,
                   edge_stable_min: float = 0.9) -> Tuple[int, int, int, int]:
    """Locate the radar panel in the top-left. Returns (x0, y0, x1, y1) in source pixels."""
    H, W = frames.shape[1:3]
    stable, detail, _ = _stability_and_detail(frames)
    sh, sw = int(H * search_frac), int(W * search_frac)
    m = ((stable[:sh, :sw] >= stable_min) & (detail[:sh, :sw] >= detail_min)).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((11, 11), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n < 2:
        raise RuntimeError("no stable, detailed region found — set CFG.crop_box by hand")
    i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    comp = (lab == i).astype(np.float32)
    rows, cols = comp.sum(1), comp.sum(0)                 # tighten to the solid core
    ry = np.where(rows >= 0.35 * rows.max())[0]
    rx = np.where(cols >= 0.35 * cols.max())[0]
    x0, y0, x1, y1 = int(rx[0]), int(ry[0]), int(rx[-1]) + 1, int(ry[-1]) + 1
    x0, y0 = max(x0 - pad, 0), max(y0 - pad, 0)
    x1, y1 = min(x1 + pad, W), min(y1 + pad, H)
    # Peel off edge lines that flicker. The panel's frame lets the 3D world bleed through, so its
    # outer few px change in ~95% of frames (measured on mirage-demo.mp4: 4px top/left).
    # Kept, they inflate the 3.1 difference % and hand the detector moving noise at the border.
    while x1 - x0 > 16 and stable[y0:y1, x0].mean() < edge_stable_min:
        x0 += 1
    while x1 - x0 > 16 and stable[y0:y1, x1 - 1].mean() < edge_stable_min:
        x1 -= 1
    while y1 - y0 > 16 and stable[y0, x0:x1].mean() < edge_stable_min:
        y0 += 1
    while y1 - y0 > 16 and stable[y1 - 1, x0:x1].mean() < edge_stable_min:
        y1 -= 1
    if square:                                            # the CS2 radar is square
        s = min(x1 - x0, y1 - y0)                         # shrink, never grow back into the frame
        cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
        x0, y0 = cx - s // 2, cy - s // 2
        x1, y1 = x0 + s, y0 + s
    return int(x0), int(y0), int(x1), int(y1)


def crop_change(frames: np.ndarray, box: Tuple[int, int, int, int]) -> float:
    """Typical fraction of crop pixels that differ from the median crop. Players and the bomb ring
    keep this at a few %; a radar that drifts, rotates or blends with the world pushes it to ~100%.
    Median over frames, so one menu / match-result frame cannot spoil it."""
    x0, y0, x1, y1 = box
    c = frames[:, y0:y1, x0:x1].astype(np.int16)
    med = np.median(c, axis=0)
    return float(np.median((np.abs(c - med).sum(-1) > 30).mean(axis=(1, 2))))


def radar_hidden(frames: np.ndarray, max_changed: float = 0.30) -> np.ndarray:
    """bool (N,) — True where the radar is covered (match-result screen, menus, pauses).
    Compares each cropped frame to the clip's median, which is the bare map."""
    c = frames.astype(np.int16)
    med = np.median(c, axis=0)
    return (np.abs(c - med).sum(-1) > 30).mean(axis=(1, 2)) > max_changed


def detect_crops(videos: Sequence[Path], cache_json: Optional[str] = None
                 ) -> Dict[str, Tuple[int, int, int, int]]:
    """Crop box per video, keyed by file name. Cached, because probing a multi-GB file is slow.

    A cache entry is reused only if the video's resolution and file size still match — replace a
    file with a new render under the same name and its box is detected again, not silently reused.
    Renaming a video keeps its size, so a renamed file finds its old entry; entries for videos no
    longer in data/ are dropped.
    A video whose radar cannot be found is left out of the result (and so out of the dataset).
    """
    cache_path = Path(cache_json or CFG.crops_json)
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    crops = {}
    for v in videos:
        res, size = list(video_resolution(v)), v.stat().st_size
        hit = cache.get(v.name) or next((e for e in cache.values()          # renamed file
                                         if e.get("bytes") == size and e["resolution"] == res), None)
        if hit and hit["resolution"] == res and hit.get("bytes") == size \
                and (CFG.crop_box is None or hit["crop_box"] == list(CFG.crop_box)):
            crops[v.name] = tuple(hit["crop_box"])
            cache[v.name] = hit                                  # re-key a renamed file
            continue
        print(f"   probing {v.name} ...")
        try:
            probe_v = sample_probe_frames(str(v), CFG.probe_frames)
            box = tuple(CFG.crop_box) if CFG.crop_box is not None \
                else find_radar_box(probe_v, CFG.search_frac)
        except RuntimeError as e:
            print(f"WARNING: {v.name} left out — {e}")
            continue
        crops[v.name] = box
        cache[v.name] = {"resolution": res, "bytes": size, "crop_box": list(box),
                         "change_pct": round(crop_change(probe_v, box) * 100, 1)}
    cache = {n: cache[n] for n in crops}                       # current videos only
    cache_path.write_text(json.dumps(cache, indent=1))

    print(f"\n{'video':<40} {'crop_box':<22} {'size':>9} {'change':>7}")
    for name, box in crops.items():
        ch = cache[name].get("change_pct", float("nan"))
        flag = "" if ch <= 10 else "   <-- check capture settings (CLAUDE.md §4)"
        print(f"{name:<40} {str(box):<22} {box[2]-box[0]:>4}x{box[3]-box[1]:<4} {ch:>6.1f}%{flag}")

    resolutions = {tuple(cache[n]["resolution"]) for n in crops}
    if len(resolutions) > 1:
        print(f"WARNING: videos were recorded at different resolutions {sorted(resolutions)}. "
              "Each gets its own crop box, but icon size relative to the map may differ — "
              "re-render at one resolution if detection quality varies between matches.")
    return crops

In [ ]:
if HAVE_VIDEO:
    CROPS = detect_crops(VIDEOS)
    dropped = [v.name for v in VIDEOS if v.name not in CROPS]
    VIDEOS = [v for v in VIDEOS if v.name in CROPS]      # only videos with a usable radar
    if dropped:
        print("left out (no radar found):", dropped)
    if Path(CFG.video_path).name not in CROPS:            # primary itself was left out
        CFG.video_path = str(VIDEOS[0]) if VIDEOS else None
    HAVE_VIDEO = bool(VIDEOS)

if HAVE_VIDEO:
    probe = sample_probe_frames(CFG.video_path, CFG.probe_frames)   # primary video, for 3.1
    CROP = CROPS.get(Path(CFG.video_path).name) or CFG.crop_box \
        or find_radar_box(probe, CFG.search_frac)
    print(f"\nprimary screen {probe.shape[2]}x{probe.shape[1]}  ->  CROP = {CROP}")
else:
    CROPS = {}
    CROP = (0, 0, CFG.minimap_px, CFG.minimap_px)
    print("synthetic mode — crop box not needed")

### 3.1 Verify the crop before trusting it

Auto-detection should never be blind. Two things to confirm by eye:

1. the crop contains the radar and nothing else (no round timer, no scoreboard);
2. the map outline is **pixel-identical** between timestamps while only the dots move.

The rightmost panel is the difference between the first and last sample. Bright spots should be
dots only. If the whole outline glows, the radar is rotating or re-centring and the capture
settings are wrong — fix `cl_radar_rotate 0` and `cl_radar_always_centered 0` and re-record,
because no amount of modelling recovers from a moving coordinate frame.

In [ ]:
def verify_crop(frames: np.ndarray, box: Tuple[int, int, int, int], n_show: int = 4) -> float:
    x0, y0, x1, y1 = box
    idx = np.linspace(0, len(frames) - 1, n_show).astype(int)
    crops = [frames[i][y0:y1, x0:x1] for i in idx]
    fig, axes = plt.subplots(1, n_show + 1, figsize=(3.1 * (n_show + 1), 3.4))
    for ax, c, i in zip(axes, crops, idx):
        ax.imshow(c); ax.set_title(f"probe {i}"); ax.axis("off")
    diff = np.abs(crops[0].astype(int) - crops[-1].astype(int)).sum(-1)
    axes[-1].imshow(diff, cmap="inferno")
    axes[-1].set_title("difference\n(dots only)"); axes[-1].axis("off")
    plt.tight_layout(); plt.show()
    changed = crop_change(frames, box)
    print(f"{changed*100:.1f}% of pixels typically differ from the median radar.")
    print("Under ~10% = locked radar (the rest is players and the bomb-radius ring).")
    return changed


def show_crop_on_screen(frame: np.ndarray, box: Tuple[int, int, int, int], name: str = ""):
    """The full replay screen with the auto-detected radar box drawn on it."""
    from matplotlib.patches import Rectangle
    x0, y0, x1, y1 = box
    fig, ax = plt.subplots(figsize=(10, 10 * frame.shape[0] / frame.shape[1]))
    ax.imshow(frame)
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="#00e676", lw=2.5))
    ax.text(x0, y1 + 40, f"radar found automatically · box {box}", color="#00e676",
            fontsize=11, weight="bold", va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="#101722", ec="none", alpha=0.85))
    ax.set_title(f"Replay screen {name} — the only part StratScout reads", fontsize=11)
    ax.axis("off"); plt.tight_layout(); plt.show()


VERIFY_ALL = False   # True -> plot every video in data/. The table printed by §3 already lists
                     # each video's change %, so only turn this on to eyeball a flagged one.

if HAVE_VIDEO:
    show_crop_on_screen(probe[len(probe) // 2], CROP, Path(CFG.video_path).stem)
    verify_crop(probe, CROP)
    if VERIFY_ALL:
        for v in VIDEOS:
            if str(v) != str(Path(CFG.video_path)):
                print(f"--- {v.name}")
                verify_crop(sample_probe_frames(str(v), CFG.probe_frames), CROPS[v.name])

## 4. Extracting frames

Rounds vary in length, but the classifier needs a fixed sequence length, so clips are sampled
at **evenly spaced** positions rather than a fixed frame rate. A 20-second round and a
40-second round both become 32 steps, which means the model sees the *shape* of an execute
rather than its duration.

In [ ]:
def extract_clip(video: str, start_s: float, end_s: float, box: Tuple[int, int, int, int],
                 n_frames: int = 32, out_px: Optional[int] = None) -> np.ndarray:
    """-> uint8 (n_frames, H, W, 3) of cropped radar."""
    assert cv2 is not None, "pip install opencv-python"
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 60.0
    total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    x0, y0, x1, y1 = box
    last = min(end_s * fps, total - 2)                    # clamp: never run off the end
    idx = np.linspace(start_s * fps, last, n_frames).astype(int)
    frames = []
    for i in idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, bgr = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[y0:y1, x0:x1]
        if out_px:
            rgb = cv2.resize(rgb, (out_px, out_px), interpolation=cv2.INTER_AREA)
        frames.append(rgb)
    cap.release()
    if len(frames) < n_frames:
        raise RuntimeError(f"got {len(frames)}/{n_frames} frames — check start/end times")
    return np.stack(frames).astype(np.uint8)


def video_duration(video: str) -> float:
    cap = cv2.VideoCapture(video)
    d = cap.get(cv2.CAP_PROP_FRAME_COUNT) / (cap.get(cv2.CAP_PROP_FPS) or 60.0)
    cap.release()
    return float(d)


def stack_resized(clips: Sequence[np.ndarray]) -> np.ndarray:
    """Stack clips from different videos. Crop boxes can differ by a pixel or two between
    renders, so anything not matching the first clip's size is resized to it (INTER_AREA)."""
    h, w = clips[0].shape[-3:-1]

    def fit(c):
        if c.shape[-3:-1] == (h, w):
            return c
        flat = c.reshape(-1, *c.shape[-3:])               # works for (N,H,W,3) and (R,T,H,W,3)
        out = np.stack([cv2.resize(f, (w, h), interpolation=cv2.INTER_AREA) for f in flat])
        return out.reshape(*c.shape[:-3], h, w, 3)

    return np.concatenate([fit(c) for c in clips]).astype(np.uint8)

In [ ]:
PILOT_TOTAL = 400   # frames pooled across every video in data/, for icon labelling


def _pilot_signature() -> List[Dict]:
    return [{"video": v.name, "bytes": v.stat().st_size} for v in VIDEOS]


def _pilot_sources_match() -> bool:
    p = Path(CFG.frames_src_json)
    if not p.exists():
        return False
    saved = json.loads(p.read_text())
    return [{"video": s["video"], "bytes": s.get("bytes")} for s in saved] == _pilot_signature()


def extract_pilot_frames() -> np.ndarray:
    per_video = math.ceil(PILOT_TOTAL / len(VIDEOS))
    clips, sources, row = [], [], 0
    for v in VIDEOS:
        dur = video_duration(str(v))
        end = CFG.end_s if CFG.end_s is not None else dur - 20
        try:
            clip = extract_clip(str(v), CFG.start_s, end, CROPS[v.name], n_frames=per_video)
        except RuntimeError as e:
            print(f"skipped {v.name}: {e}")
            continue
        hidden = radar_hidden(clip)                       # match-result screen, menus, pauses
        clip = clip[~hidden]
        sources.append({"match_id": match_id_of(v), "video": v.name,
                        "bytes": v.stat().st_size, "rows": [row, row + len(clip)]})
        clips.append(clip); row += len(clip)
        print(f"{v.name:<40} {dur:5.0f}s  ->  {len(clip)} frames from {CFG.start_s:.0f}s to "
              f"{end:.0f}s" + (f"   ({hidden.sum()} dropped: radar hidden)" if hidden.any() else ""))
    frames = stack_resized(clips)
    np.save(CFG.frames_npy, frames)
    Path(CFG.frames_src_json).write_text(json.dumps(sources, indent=1))
    print("saved", CFG.frames_npy, frames.shape, frames.dtype,
          "| row ranges per match ->", CFG.frames_src_json)
    return frames


if not HAVE_VIDEO:
    pilot_frames = None
    print("synthetic mode — no frames extracted")
elif Path(CFG.icon_labels_json).exists() and Path(CFG.frames_npy).exists():
    # labels index into the saved frames — re-extracting would silently scramble them
    pilot_frames = np.load(CFG.frames_npy)
    print(f"{CFG.icon_labels_json} exists — reusing {CFG.frames_npy} {pilot_frames.shape}. "
          "New videos are NOT added to the icon set; delete both files to rebuild it.")
elif Path(CFG.frames_npy).exists() and _pilot_sources_match():
    pilot_frames = np.load(CFG.frames_npy)
    print(f"reusing {CFG.frames_npy} {pilot_frames.shape} — same videos as last time")
else:
    if Path(CFG.frames_npy).exists():                     # never destroy frames: move aside
        bak = Path(CFG.frames_npy).with_suffix(f".bak-{datetime.now():%Y%m%d-%H%M%S}.npy")
        Path(CFG.frames_npy).rename(bak)
        print(f"videos changed — old frames kept as {bak.name}")
    pilot_frames = extract_pilot_frames()

## 5. Finding icons without a neural network

Before training anything, a plain colour rule finds most icons: the map is painted in greys and
muted blues, the icons are vivid **orange (T)** and **blue (CT)**. High saturation separates
them almost perfectly.

Three refinements that matter in practice:

* **Static-background subtraction.** Take the median frame — the map with players washed out,
  like a long-exposure photo of a street where the buildings are sharp and the pedestrians
  vanish. Keep only pixels that *differ* from it. This removes the green spawn outlines and the
  A/B site letters in one step, with no hand-placed exclusion boxes.
* **Saturation-weighted centres.** Each icon carries a pale facing-cone that drags a plain
  centroid off-target, so pixels are weighted by saturation² and the vivid core wins.
* **Learned team hues.** Measured from your own footage rather than guessed.

This classical finder does two jobs: it proposes labels so you correct rather than click from
scratch, and it becomes the **baseline the CNN has to beat** — a far stronger result in a
report than a CNN accuracy quoted on its own.

In [ ]:
DEFAULT_TEAM_HUES = {"T": 17.0, "CT": 105.0}     # OpenCV hue 0-179: orange, blue


def _hue_dist(a: float, b: float) -> float:
    d = abs(a - b)
    return min(d, 180 - d)


def learn_background(frames: np.ndarray, sample: int = 150) -> np.ndarray:
    """Median frame: the map, with the players removed."""
    idx = np.linspace(0, len(frames) - 1, min(sample, len(frames))).astype(int)
    return np.median(frames[idx].astype(np.float32), axis=0).astype(np.uint8)


def learn_team_hues(frames: np.ndarray, sample: int = 60,
                    sat_min: int = None, val_min: int = None) -> Dict[str, float]:
    """Measure the two real hues from the footage instead of trusting the defaults."""
    sat_min = CFG.sat_min if sat_min is None else sat_min
    val_min = CFG.val_min if val_min is None else val_min
    idx = np.linspace(0, len(frames) - 1, min(sample, len(frames))).astype(int)
    hues: List[float] = []
    for i in idx:
        hsv = cv2.cvtColor(frames[int(i)], cv2.COLOR_RGB2HSV)
        m = (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)
        hues += hsv[:, :, 0][m].astype(float).tolist()
    if not hues:
        raise RuntimeError("no saturated pixels — lower sat_min/val_min")
    hist = np.bincount(np.round(hues).astype(int) % 180, minlength=180).astype(float)
    k = np.array([1., 2., 3., 2., 1.]); k /= k.sum()
    sm = np.convolve(np.r_[hist[-2:], hist, hist[:2]], k, mode="same")[2:-2]
    h = np.arange(180)
    warm = int(np.argmax(np.where(h < 45, sm, 0)))                       # orange band
    cool = int(np.argmax(np.where((h > 80) & (h < 135), sm, 0)))         # blue band
    print(f"learned team hues:  T (orange) ~{warm}   CT (blue) ~{cool}")
    return {"T": float(warm), "CT": float(cool)}


def find_icons_teams(frame: np.ndarray, team_hues: Optional[Dict[str, float]] = None,
                     background: Optional[np.ndarray] = None,
                     sat_min: int = None, val_min: int = None,
                     area: Tuple[int, int] = None, hue_tol: float = None,
                     bg_thresh: int = None, max_per_team: int = None) -> List[List[float]]:
    """-> [[x, y, team], ...] with x,y normalised and team 0 = T (orange), 1 = CT (blue)."""
    hues = team_hues or DEFAULT_TEAM_HUES
    sat_min = CFG.sat_min if sat_min is None else sat_min
    val_min = CFG.val_min if val_min is None else val_min
    area = CFG.icon_area if area is None else area
    hue_tol = CFG.hue_tol if hue_tol is None else hue_tol
    bg_thresh = CFG.bg_thresh if bg_thresh is None else bg_thresh
    max_per_team = CFG.n_per_team if max_per_team is None else max_per_team

    h, w = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_RGB2HSV)
    mask = (hsv[:, :, 1] >= sat_min) & (hsv[:, :, 2] >= val_min)
    if background is not None:
        mask &= np.abs(frame.astype(np.int16) - background.astype(np.int16)).sum(-1) > bg_thresh
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    found: List[Tuple[int, float, List[float]]] = []
    for i in range(1, n):
        a = int(stats[i, cv2.CC_STAT_AREA])
        if not (area[0] <= a <= area[1]):
            continue
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        if max(bw, bh) > 3 * max(min(bw, bh), 1):          # drop thin streaks (map outlines)
            continue
        ys, xs = np.nonzero(lab == i)
        sat = hsv[ys, xs, 1].astype(np.float32)
        wt = sat ** 2                                      # vivid core beats the pale cone
        x = float((xs * wt).sum() / wt.sum()) / w
        y = float((ys * wt).sum() / wt.sum()) / h
        core = sat >= np.percentile(sat, 60)
        hue = float(np.median(hsv[ys, xs, 0][core]))
        best = min(CFG.teams, key=lambda t: _hue_dist(hue, hues[t]))
        if _hue_dist(hue, hues[best]) > hue_tol:
            continue                                       # not orange or blue: bomb, X, marker
        found.append((CFG.teams.index(best), float(a), [x, y]))

    out: List[List[float]] = []
    for ti in (0, 1):
        same = sorted([f for f in found if f[0] == ti], key=lambda f: -f[1])[:max_per_team]
        out += [[p[0], p[1], float(ti)] for _, _, p in same]
    return out

In [ ]:
if HAVE_VIDEO:
    BG = learn_background(pilot_frames)
    HUES = learn_team_hues(pilot_frames)
    _d = find_icons_teams(pilot_frames[0], HUES, BG)
    print(f"frame 0: {sum(p[2]==0 for p in _d)} T, {sum(p[2]==1 for p in _d)} CT")
else:
    BG, HUES = None, DEFAULT_TEAM_HUES
    print("synthetic mode — icon finder will be exercised in Section 7")

# Which radar the synthetic rounds get drawn on. Your own footage wins: its median frame IS
# the real Mirage radar, at the right scale and colours, with the players washed out.
MAP_BG_SOURCE = BG if BG is not None else (
    CFG.map_image if Path(CFG.map_image).exists() else None)
print("synthetic background:",
      "median of your footage" if isinstance(MAP_BG_SOURCE, np.ndarray)
      else (f"map image {CFG.map_image}" if MAP_BG_SOURCE else "drawn abstraction (no real map available)"))
print("plot background:     ",
      f"{CFG.map_image}" if Path(CFG.map_image).exists() else "same as synthetic background")

## 6. Seeing what the finder found

The most useful debugging tool in this project is a picture. Three views:

* **`show_detections`** — icons circled on a strip of frames, orange for T and blue for CT.
* **`show_mask_pipeline`** — frame, saturation, mask, result, so a bad threshold is visible
  rather than inferred from a count.
* **`show_counts`** — how many icons per frame across the whole clip. Healthy data sits near
  5 and 5 and dips as players die; a flat line at 1–2 means the finder is broken, not that
  everyone is dead.

In [ ]:
TEAM_EDGE = {0: "#ff9020", 1: "#40a0ff"}


def show_detections(frames: np.ndarray, idxs: Sequence[int] = None, n: int = 6,
                    team_hues=None, background=None, title: str = "classical icon finder"):
    """Circle every detected icon on a strip of frames (orange = T, blue = CT)."""
    idxs = idxs if idxs is not None else np.linspace(0, len(frames) - 1, n).astype(int)
    fig, axes = plt.subplots(1, len(idxs), figsize=(3.0 * len(idxs), 3.5))
    for ax, i in zip(np.atleast_1d(axes), idxs):
        f = frames[int(i)]
        pts = find_icons_teams(f, team_hues, background)
        ax.imshow(f)
        h, w = f.shape[:2]
        for x, y, t in pts:
            ax.plot(x * w, y * h, "o", mfc="none", mec=TEAM_EDGE[int(t)], ms=13, mew=2)
        nT = sum(int(p[2]) == 0 for p in pts)
        ax.set_title(f"f{i} — {nT}T / {len(pts)-nT}CT", fontsize=10)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout(); plt.show()


def show_mask_pipeline(frames: np.ndarray, i: int = 0, team_hues=None, background=None):
    """Frame -> saturation -> mask -> detections, so threshold problems are visible."""
    f = frames[int(i)]
    hsv = cv2.cvtColor(f, cv2.COLOR_RGB2HSV)
    mask = (hsv[:, :, 1] >= CFG.sat_min) & (hsv[:, :, 2] >= CFG.val_min)
    if background is not None:
        mask &= np.abs(f.astype(np.int16) - background.astype(np.int16)).sum(-1) > CFG.bg_thresh
    pts = find_icons_teams(f, team_hues, background)
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    axes[0].imshow(f); axes[0].set_title(f"frame {i}")
    axes[1].imshow(hsv[:, :, 1], cmap="viridis"); axes[1].set_title("saturation")
    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title(f"mask (S>={CFG.sat_min}, V>={CFG.val_min}{', bg-sub' if background is not None else ''})")
    axes[3].imshow(f); axes[3].set_title(f"found {len(pts)}")
    h, w = f.shape[:2]
    for x, y, t in pts:
        axes[3].plot(x * w, y * h, "o", mfc="none", mec=TEAM_EDGE[int(t)], ms=13, mew=2)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout(); plt.show()
    print("too many blobs -> raise sat_min or narrow icon_area | missing icons -> lower them")


def show_counts(frames: np.ndarray, team_hues=None, background=None, step: int = 1):
    idx = range(0, len(frames), step)
    nT, nC = [], []
    for i in idx:
        pts = find_icons_teams(frames[i], team_hues, background)
        nT.append(sum(int(p[2]) == 0 for p in pts))
        nC.append(len(pts) - nT[-1])
    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.plot(list(idx), nT, lw=1.2, color=TEAM_EDGE[0], label="T (orange)")
    ax.plot(list(idx), nC, lw=1.2, color=TEAM_EDGE[1], label="CT (blue)")
    ax.axhline(5, ls="--", c="grey", lw=1)
    ax.set_xlabel("frame"); ax.set_ylabel("icons found"); ax.set_ylim(-0.3, 6.5)
    ax.legend(fontsize=8); ax.set_title("icons per frame — should hover near 5 and dip on deaths")
    plt.tight_layout(); plt.show()
    print(f"mean  T {np.mean(nT):.2f}   CT {np.mean(nC):.2f}   "
          f"frames with 5T+5CT: {sum(a == 5 and b == 5 for a, b in zip(nT, nC))}/{len(nT)}")

In [ ]:
if HAVE_VIDEO:
    show_mask_pipeline(pilot_frames, 0, HUES, BG)
    show_detections(pilot_frames, team_hues=HUES, background=BG)
    show_counts(pilot_frames, HUES, BG, step=4)

## 7. Synthetic fallback

A stand-in generator so every cell below runs without footage. It fakes a small map with a T
spawn bottom, A site top-right, B site top-left and mid in the middle, then walks orange
attackers along class-appropriate routes while blue defenders hold angles.

The distinction the classifier must learn:

Before the plant (attackers):

* **RUSH** — all five attackers, one route, fast.
* **SPLIT** — three on one route, two on another, converging.
* **DEFAULT** — slow map control: spread over the map, no convergence on a site.

After the plant (defenders, only if the bomb went down):

* **RETAKE** — the defenders regroup and arrive on the site together.
* **TRICKLE** — they arrive one at a time, so each fights alone.
* **SAVE** — they give the site up and move away to keep their weapons.

Every generated player stays on walkable map floor (`MAP_MASK`).

**The radar is the real one.** Rounds are drawn on the median frame of your own footage when
you have it, or on a Mirage radar PNG (`Config.map_image`) otherwise, and the two bombsites
and two spawns are measured off the real map rather than invented. The connecting waypoints
are estimates — run `pick_map_points(_bg)` once to click your own and they are reused from
`map_points.json` from then on.

Replace this whole section with real rounds anyway. Everything downstream consumes `Round`
objects and does not care where they came from.

In [ ]:
# Real Mirage callouts, normalised to the radar crop. The two sites and two spawns are
# MEASURED (bombsite outlines and spawn zones on the Mirage radar; B matches the 0.230, 0.276
# you read off your own footage). The connecting waypoints are ESTIMATES of the routes between
# them — run pick_map_points() to replace them by clicking on your own radar.
MAP_POINTS = {
    "t_spawn":   (0.880, 0.360),   # measured - green spawn zone, right
    "ct_spawn":  (0.297, 0.697),   # measured - green spawn zone, lower left
    "a_site":    (0.545, 0.770),   # measured - orange bombsite outline, lower middle
    "b_site":    (0.230, 0.276),   # measured - orange bombsite outline, upper left
    "mid":       (0.520, 0.470),   # estimate
    "a_ramp":    (0.800, 0.630),   # estimate - T spawn down towards A
    "a_approach": (0.660, 0.745),  # estimate
    "connector": (0.620, 0.580),   # estimate - mid into A
    "b_apps":    (0.520, 0.215),   # estimate - upper route to B
    "b_short":   (0.380, 0.330),   # estimate - mid into B
}

MAP_POINTS_FILE = "map_points.json"
if Path(MAP_POINTS_FILE).exists():
    MAP_POINTS = {k: tuple(v) for k, v in json.loads(Path(MAP_POINTS_FILE).read_text()).items()}
    print(f"loaded callouts from {MAP_POINTS_FILE}")

A_SITE = np.array(MAP_POINTS["a_site"], np.float32)
B_SITE = np.array(MAP_POINTS["b_site"], np.float32)

T_ROUTES = {
    "A_RUSH":  [["t_spawn", "a_ramp", "a_approach", "a_site"]] * 5,
    "B_RUSH":  [["t_spawn", "b_apps", "b_site"]] * 5,
    "A_SPLIT": [["t_spawn", "a_ramp", "a_approach", "a_site"]] * 3 +
               [["t_spawn", "mid", "connector", "a_site"]] * 2,
    "B_SPLIT": [["t_spawn", "b_apps", "b_site"]] * 3 +
               [["t_spawn", "mid", "b_short", "b_site"]] * 2,
    # slow map control: spread over the map, no convergence on either site
    "DEFAULT": [["t_spawn", "mid"], ["t_spawn", "b_apps"], ["t_spawn", "a_ramp"],
                ["t_spawn", "mid", "connector"], ["t_spawn", "a_ramp", "a_approach"]],
}
RUSH_SPEED = {"A_RUSH": (0.30, 0.45), "B_RUSH": (0.30, 0.45),
              "A_SPLIT": (0.60, 0.90), "B_SPLIT": (0.60, 0.90), "DEFAULT": (0.80, 1.00)}
CT_ANCHORS = ["a_site", "a_approach", "mid", "b_short", "b_site"]

_GRID_CACHE: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}


def _grid(size: int):
    if size not in _GRID_CACHE:
        yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
        _GRID_CACHE[size] = (xx.astype(np.float32), yy.astype(np.float32))
    return _GRID_CACHE[size]


def _polyline(wp: np.ndarray, s: np.ndarray) -> np.ndarray:
    wp = np.asarray(wp, np.float64)
    seg = np.linalg.norm(np.diff(wp, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    d = np.clip(s, 0, 1) * max(cum[-1], 1e-9)
    i = np.clip(np.searchsorted(cum, d, side="right") - 1, 0, len(seg) - 1)
    frac = (d - cum[i]) / np.maximum(seg[i], 1e-9)
    return wp[i] + (wp[i + 1] - wp[i]) * frac[:, None]


def _jitter_on_map(p, sd: float, rng: np.random.Generator, tries: int = 40) -> np.ndarray:
    """Random offset around p, re-drawn until it lands on walkable map (never on black)."""
    p = np.asarray(p, np.float64)
    size = MAP_MASK.shape[0]
    for _ in range(tries):
        q = np.clip(p + rng.normal(0, sd, 2), 0.0, 0.999)
        if MAP_MASK[int(q[1] * size), int(q[0] * size)]:
            return q
    return snap_to_map(p[None].astype(np.float32))[0].astype(np.float64)


def _walk(names, n, rng, delay=0.0, duration=1.0, jitter=0.008):
    wp = np.array([MAP_POINTS[x] for x in names], np.float64)
    for k in range(1, len(wp) - 1):
        wp[k] = _jitter_on_map(wp[k], 0.02, rng)
    wp[0] = _jitter_on_map(wp[0], 0.06, rng)     # five players do not stand on one pixel at spawn
    wp[-1] = _jitter_on_map(wp[-1], 0.06, rng)   # and they fan out on arrival
    tt = np.linspace(0, 1, n)
    s = np.clip((tt - delay) / max(duration, 1e-6), 0, 1)
    s = s * s * (3 - 2 * s)
    return np.clip(_polyline(wp, s) + rng.normal(0, jitter, (n, 2)), 0.03, 0.97)


def _hold(anchor, n, rng, amp=0.012):
    drift = np.cumsum(rng.normal(0, amp, (n, 2)), axis=0) * 0.5
    return np.clip(np.array(anchor, np.float64)[None] + drift + rng.normal(0, 0.006, (n, 2)),
                   0.03, 0.97)


def make_round(label: str, rng: np.random.Generator, n: int) -> np.ndarray:
    """Before the plant: the attackers run their tactic while the defenders hold their spots."""
    pos = np.zeros((n, N_PLAYERS, 2), np.float32)
    lo, hi = RUSH_SPEED[label]
    late = 0.12 if "RUSH" in label else 0.30 if label == "DEFAULT" else 0.25
    for p, route in enumerate(T_ROUTES[label]):
        pos[:, p] = _walk(route, n, rng, rng.uniform(0.0, late), rng.uniform(lo, hi))
    for p, a in enumerate(CT_ANCHORS):
        pos[:, 5 + p] = _hold(MAP_POINTS[a], n, rng)
    return snap_to_map(pos)


# After the plant. Where the defenders go when they give the site up.
SAVE_POINTS = {"A": ["b_site", "b_short", "mid"], "B": ["a_approach", "connector", "a_site"]}


def _walk_xy(wp, n, rng, delay=0.0, duration=1.0, jitter=0.008) -> np.ndarray:
    """Like _walk, but through explicit (x, y) waypoints instead of callout names."""
    wp = np.asarray(wp, np.float64)
    tt = np.linspace(0, 1, n)
    s = np.clip((tt - delay) / max(duration, 1e-6), 0, 1)
    s = s * s * (3 - 2 * s)
    return np.clip(_polyline(wp, s) + rng.normal(0, jitter, (n, 2)), 0.03, 0.97)


def make_post_plant(response: str, site: str, start: np.ndarray, rng: np.random.Generator,
                    n: int) -> np.ndarray:
    """After the plant, (n, 10, 2). `start` is the last frame before the plant, (10, 2).
    RETAKE  - the defenders regroup and arrive on the site together.
    TRICKLE - they arrive one at a time, so each fights alone.
    SAVE    - they give the site up and move away to keep their weapons."""
    pos = np.zeros((n, N_PLAYERS, 2), np.float32)
    site_xy = np.array(MAP_POINTS["a_site" if site == "A" else "b_site"])
    for p in range(CFG.n_per_team):                       # attackers guard the planted bomb
        pos[:, p] = _hold(_jitter_on_map(site_xy, 0.035, rng), n, rng, 0.008)
    stagger = np.sort(rng.uniform(0.0, 0.55, CFG.n_per_team))
    for k in range(CFG.n_per_team):
        s0 = start[CFG.n_per_team + k].astype(np.float64)
        goal = _jitter_on_map(site_xy, 0.03, rng)
        via = [np.array(MAP_POINTS["ct_spawn"])] if np.linalg.norm(s0 - site_xy) > 0.25 else []
        if response == "RETAKE":
            path, delay, dur = [s0, *via, goal], rng.uniform(0.25, 0.35), rng.uniform(0.40, 0.50)
        elif response == "TRICKLE":
            path, delay, dur = [s0, *via, goal], stagger[k], rng.uniform(0.25, 0.40)
        else:
            away = _jitter_on_map(MAP_POINTS[SAVE_POINTS[site][k % 3]], 0.03, rng)
            path, delay, dur = [s0, away], rng.uniform(0.0, 0.25), rng.uniform(0.5, 0.8)
        pos[:, CFG.n_per_team + k] = _walk_xy(path, n, rng, delay, dur)
    return snap_to_map(pos)


def _drawn_background(size: int, rng: np.random.Generator) -> np.ndarray:
    """Last-resort abstraction: corridors drawn between callouts. Used only when neither real
    footage nor a map image is available."""
    bg = np.full((size, size, 3), (14, 18, 26), np.float32)
    corridors = [["t_spawn", "a_ramp", "a_approach", "a_site"], ["t_spawn", "b_apps", "b_site"],
                 ["t_spawn", "mid", "ct_spawn"], ["mid", "connector", "a_site"],
                 ["mid", "b_short", "b_site"], ["a_site", "a_approach", "mid"]]
    s_ = np.linspace(0, 1, 400)
    for names in corridors:
        pts = _polyline(np.array([MAP_POINTS[x] for x in names]), s_) * size
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                xs = np.clip(pts[:, 0].astype(int) + dx, 0, size - 1)
                ys = np.clip(pts[:, 1].astype(int) + dy, 0, size - 1)
                bg[ys, xs] = (46, 52, 64)
    for name in ("a_site", "b_site"):
        cx, cy = (np.array(MAP_POINTS[name]) * size).astype(int)
        r = size // 12
        bg[max(cy - r, 0):cy + r, max(cx - r, 0):cx + r] = (58, 64, 78)
    return np.clip(bg + rng.normal(0, 3.0, bg.shape), 0, 255)


def map_image_background(path: str, size: int, dark=(12, 16, 24)) -> np.ndarray:
    """Load a Mirage radar PNG and composite it over a dark panel, so it looks like the
    in-game radar rather than a white-background diagram."""
    raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise FileNotFoundError(path)
    if raw.shape[2] == 4:
        rgb = cv2.cvtColor(raw[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
        a = (raw[:, :, 3:4].astype(np.float32) / 255.0)
    else:
        rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB).astype(np.float32)
        a = (rgb.min(-1, keepdims=True) < 235).astype(np.float32)   # treat white as transparent
    comp = rgb * a + np.array(dark, np.float32) * (1 - a)
    return cv2.resize(comp, (size, size), interpolation=cv2.INTER_AREA)


def build_background(size: int, rng: np.random.Generator, source=None) -> np.ndarray:
    """The radar the synthetic rounds are drawn on.

    Preference order, best first:
      1. `source` as an array  -> the median frame of YOUR footage: the real radar, exactly as
         the detector will see it, with players washed out. Nothing to download, nothing to
         align.
      2. `source` as a path    -> a Mirage radar image, composited over a dark panel.
      3. neither               -> the drawn abstraction.
    """
    if isinstance(source, np.ndarray):
        bg = cv2.resize(source.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
        return np.clip(bg + rng.normal(0, 2.0, bg.shape), 0, 255)
    if isinstance(source, str) and Path(source).exists():
        bg = map_image_background(source, size)
        return np.clip(bg + rng.normal(0, 2.0, bg.shape), 0, 255)
    return _drawn_background(size, rng)


def plot_background() -> np.ndarray:
    """Background for every PLOT (never for synthetic training frames). Best first:
      1. CFG.map_image at full resolution — the clean radar screenshot. It frames the same area
         as the crop (checked against the footage median: correlation 0.986 on a plain resize).
      2. the median of your footage, 3. the drawn abstraction.
    Synthetic frames keep using build_background: the screenshot has the orange bomb icon baked
    in, and a fixed T-coloured blob in training frames would teach the detector the wrong thing."""
    p = Path(CFG.map_image)
    if p.exists():
        n = max(cv2.imread(str(p), cv2.IMREAD_UNCHANGED).shape[:2])
        return map_image_background(str(p), n).astype(np.uint8)
    return build_background(CFG.minimap_px, np.random.default_rng(1234),
                            MAP_BG_SOURCE).astype(np.uint8)


def pick_map_points(background: np.ndarray, out_json: str = "map_points.json",
                    names: Sequence[str] = None):
    """Click each callout once on the real radar, in the order printed. Saves map_points.json,
    which is picked up automatically next run."""
    names = list(names or MAP_POINTS.keys())
    pts: Dict[str, Tuple[float, float]] = {}
    h, w = background.shape[:2]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(background.astype(np.uint8))

    def title():
        ax.set_title(f"click: {names[len(pts)]}" if len(pts) < len(names) else "done - close me")
        fig.canvas.draw_idle()

    def on_click(ev):
        if ev.inaxes != ax or ev.xdata is None or len(pts) >= len(names):
            return
        pts[names[len(pts)]] = (round(float(ev.xdata) / w, 3), round(float(ev.ydata) / h, 3))
        ax.plot(ev.xdata, ev.ydata, "o", mfc="none", mec="#00ff88", ms=12, mew=2)
        if len(pts) == len(names):
            Path(out_json).write_text(json.dumps(pts, indent=1))
            print(f"saved {len(pts)} callouts -> {out_json}")
        title()

    fig.canvas.mpl_connect("button_press_event", on_click)
    title(); plt.show()
    return pts


def walkable_mask(bg: np.ndarray, dark: float = 25.0) -> np.ndarray:
    """bool (H, W): True on the map's floor. The radar is near-black off the map and inside
    buildings, and no player can ever stand there."""
    m = (bg.astype(np.float32).mean(-1) > dark).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))      # drop noise specks
    return cv2.dilate(m, np.ones((3, 3), np.uint8)).astype(bool)            # icons at the edge


def _nearest_walkable(mask: np.ndarray) -> np.ndarray:
    """(H, W, 2) pixel x, y of the nearest walkable pixel, for every pixel."""
    _, lab = cv2.distanceTransformWithLabels((~mask).astype(np.uint8), cv2.DIST_L2, 5,
                                             labelType=cv2.DIST_LABEL_PIXEL)
    ys, xs = np.where(mask)
    lut = np.zeros((lab.max() + 1, 2), np.float32)
    lut[lab[ys, xs]] = np.stack([xs, ys], 1)
    return lut[lab]


MAP_MASK = walkable_mask(build_background(CFG.minimap_px, np.random.default_rng(1234),
                                          MAP_BG_SOURCE))
_NEAREST = _nearest_walkable(MAP_MASK)
print(f"walkable map: {MAP_MASK.mean() * 100:.0f}% of the radar crop")


def snap_to_map(pos: np.ndarray) -> np.ndarray:
    """Move any synthetic player that landed off the map onto the nearest walkable pixel.
    Spawn and arrival jitter used to push attackers onto the black area beside T spawn."""
    size = MAP_MASK.shape[0]
    ix = np.clip((pos[..., 0] * size).astype(int), 0, size - 1)
    iy = np.clip((pos[..., 1] * size).astype(int), 0, size - 1)
    off = ~MAP_MASK[iy, ix]
    out = pos.copy()
    out[off] = (_NEAREST[iy[off], ix[off]] + 0.5) / size
    return out


T_COLOUR = np.array([255, 150, 40], np.float32)     # orange attackers
CT_COLOUR = np.array([70, 150, 255], np.float32)    # blue defenders


def render_frames(pos, bg, size, rng, radius=1.7) -> np.ndarray:
    xx, yy = _grid(size)
    out = np.empty((pos.shape[0], size, size, 3), np.uint8)
    colours = np.stack([T_COLOUR] * CFG.n_per_team + [CT_COLOUR] * CFG.n_per_team)
    for f in range(pos.shape[0]):
        px, py = pos[f, :, 0] * size, pos[f, :, 1] * size
        d2 = (xx[None] - px[:, None, None]) ** 2 + (yy[None] - py[:, None, None]) ** 2
        blob = np.exp(-d2 / (2 * radius ** 2))
        img = bg + np.einsum("psq,pc->sqc", blob, colours) + rng.normal(0, 3.0, bg.shape)
        out[f] = np.clip(img, 0, 255).astype(np.uint8)
    return out


_rng = np.random.default_rng(CFG.seed)
_bg = build_background(CFG.minimap_px, np.random.default_rng(1234), MAP_BG_SOURCE)
_demo = render_frames(make_round("A_SPLIT", _rng, CFG.seq_len), _bg, CFG.minimap_px, _rng)
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, f in zip(axes, [0, 10, 20, 31]):
    ax.imshow(_demo[f]); ax.set_title(f"t={f}"); ax.axis("off")
fig.suptitle("Synthetic radar — A_SPLIT (two attacker routes converging on A)")
plt.tight_layout(); plt.show()

# the same visualiser from Section 6, applied to synthetic frames
show_detections(_demo, idxs=[0, 8, 16, 24, 31], title="icon finder on synthetic frames")

## 8. Rounds: the unit of labelled data

One `Round` = one labelled sample. `positions` is the oracle (parsed `.dem`, or synthetic
truth); `frames` is what the deployed pipeline actually sees. `match_id` exists because splits
must be **by match, never by frame** — frames inside a round are near-copies of each other, and
shuffling them is revising with the exam paper.

In [ ]:
@dataclass
class Round:
    match_id: str
    round_idx: int
    label: str                                    # attacker tactic before the plant (T_CLASSES)
    positions: Optional[np.ndarray] = None        # (T, 10, 2) before the plant, oracle
    frames: Optional[np.ndarray] = None           # (T, H, W, 3) before the plant, pixels
    winner: Optional[str] = None                  # "T" or "CT" - needed for Objective 3
    plant_site: Optional[str] = None              # "A" / "B"; None = the bomb was never planted
    ct_label: Optional[str] = None                # defender response after the plant (CT_CLASSES)
    post_positions: Optional[np.ndarray] = None   # (T, 10, 2) after the plant, oracle
    post_frames: Optional[np.ndarray] = None      # (T, H, W, 3) after the plant, pixels

    @property
    def y(self) -> int:
        return T_CLASSES.index(self.label)

    @property
    def planted(self) -> bool:
        return self.plant_site is not None

    def label_of(self, phase: str) -> Optional[str]:
        return self.label if phase == "pre" else self.ct_label

    def positions_of(self, phase: str) -> Optional[np.ndarray]:
        return self.positions if phase == "pre" else self.post_positions

    def frames_of(self, phase: str) -> Optional[np.ndarray]:
        return self.frames if phase == "pre" else self.post_frames

    @property
    def tactic_worked(self) -> Optional[bool]:
        """The attackers' tactic worked if T won the round."""
        return None if self.winner is None else self.winner == "T"

    @property
    def response_worked(self) -> Optional[bool]:
        """The defenders' response worked if CT won a round in which the bomb was planted."""
        return None if self.winner is None or not self.planted else self.winner == "CT"


# Synthetic outcomes: invented rates so Objective 3 has something to aggregate. Not findings.
P_PLANT = {"A_RUSH": 0.80, "B_RUSH": 0.80, "A_SPLIT": 0.75, "B_SPLIT": 0.75, "DEFAULT": 0.55}
P_CT_WIN_AFTER_PLANT = {"RETAKE": 0.45, "TRICKLE": 0.25, "SAVE": 0.03}
P_T_WIN_NO_PLANT = 0.30
SITE_OF = {"A_RUSH": "A", "A_SPLIT": "A", "B_RUSH": "B", "B_SPLIT": "B"}


def generate_split(n_matches: int, prefix: str, seed: int, render: bool,
                   rounds_per_match: int = 20) -> List[Round]:
    rng = np.random.default_rng(seed)
    bg = build_background(CFG.minimap_px, np.random.default_rng(1234), MAP_BG_SOURCE)
    out = []
    for m in range(n_matches):
        t_w = rng.dirichlet(np.ones(len(T_CLASSES)) * 2.5)        # teams have tendencies
        ct_w = rng.dirichlet(np.ones(len(CT_CLASSES)) * 2.5)
        for r in range(rounds_per_match):
            label = T_CLASSES[int(rng.choice(len(T_CLASSES), p=t_w))]
            pos = make_round(label, rng, CFG.seq_len)
            site = ct = post = None
            if rng.random() < P_PLANT[label]:
                site = SITE_OF.get(label) or ("A" if rng.random() < 0.5 else "B")
                ct = CT_CLASSES[int(rng.choice(len(CT_CLASSES), p=ct_w))]
                post = make_post_plant(ct, site, pos[-1], rng, CFG.seq_len)
                winner = "CT" if rng.random() < P_CT_WIN_AFTER_PLANT[ct] else "T"
            else:
                winner = "T" if rng.random() < P_T_WIN_NO_PLANT else "CT"
            rd = Round(f"{prefix}_m{m:03d}", r, label, pos, None, winner, site, ct, post)
            if render:
                rd.frames = render_frames(pos, bg, CFG.minimap_px, rng)
                if post is not None:
                    rd.post_frames = render_frames(post, bg, CFG.minimap_px, rng)
            out.append(rd)
    return out


train_rounds = generate_split(12, "train", CFG.seed + 1, render=False)
val_rounds = generate_split(3, "val", CFG.seed + 2, render=False)
test_rounds = generate_split(3, "test", CFG.seed + 3, render=True)
det_pool = generate_split(3, "detpool", CFG.seed + 4, render=True)
det_pool_val = generate_split(1, "detval", CFG.seed + 5, render=True)

ids = {n: {r.match_id for r in rs} for n, rs in
       [("train", train_rounds), ("val", val_rounds), ("test", test_rounds)]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) \
    and not (ids["val"] & ids["test"]), "match-id leakage between splits"
print(f"train {len(train_rounds)} | val {len(val_rounds)} | test {len(test_rounds)} rounds "
      f"- no match-id overlap")
print("attacker tactics (train):", {c: sum(r.label == c for r in train_rounds) for c in T_CLASSES})
print(f"bomb planted in {sum(r.planted for r in train_rounds)}/{len(train_rounds)} train rounds; "
      "defender responses:", {c: sum(r.ct_label == c for r in train_rounds) for c in CT_CLASSES})

## 9. Pre-processing

Crop, resize, scale to 0–1, move channels first. PyTorch wants `(batch, channels, h, w)`;
images arrive as `(h, w, channels)`.

In [ ]:
class MinimapPreprocessor(nn.Module):
    def __init__(self, out_px: int):
        super().__init__()
        self.out_px = out_px

    def forward(self, frames_hwc) -> torch.Tensor:
        x = torch.as_tensor(np.ascontiguousarray(frames_hwc))
        if x.ndim == 3:
            x = x[None]
        x = x.permute(0, 3, 1, 2).float() / 255.0
        if x.shape[-1] != self.out_px or x.shape[-2] != self.out_px:
            x = F.interpolate(x, size=(self.out_px, self.out_px), mode="bilinear",
                              align_corners=False)
        return x


PRE = MinimapPreprocessor(CFG.minimap_px)
check_contract("frames", PRE(test_rounds[0].frames),
               (CFG.seq_len, 3, CFG.minimap_px, CFG.minimap_px))

## 10. Model 1 — the icon detector

A **fully convolutional centre-heatmap detector** (CenterNet family): seven 3×3 convolutions
and a 1×1 output head, ~98k parameters, no bounding boxes anywhere.

Boxes are skipped deliberately. Every icon is the same size, so a box adds nothing a centre
does not; the tracker consumes points, not boxes; and clicking a centre is much faster to
annotate than dragging a rectangle.

The output is two greyscale images — one per team — where brightness means "an icon is here".
Like a weather map, where rain means a player. Channel 0 is T, channel 1 is CT, so **team
classification comes free** from which map the peak appears in.

Two details that are easy to get wrong and expensive to debug:

* **The Gaussian is centred on an integer pixel and that pixel is forced to 1.0.** Centre it on
  the true sub-pixel location instead and the brightest pixel may only reach ~0.8, the focal
  loss then finds no positive pixels at all, and the detector silently learns to predict
  nothing. That looks like a broken model; it is a broken label.
* **Focal loss, not cross-entropy.** About 99.9% of pixels are background, so plain
  cross-entropy is minimised by predicting "nothing, anywhere".

In [ ]:
class MinimapDetector(nn.Module):
    def __init__(self, n_classes: int = 2, width: int = 32, out_stride: int = None):
        super().__init__()
        out_stride = CFG.det_stride if out_stride is None else out_stride
        w = width
        self.out_stride = out_stride
        self.enc = nn.Sequential(
            nn.Conv2d(3, w, 3, 2, 1), nn.BatchNorm2d(w), nn.ReLU(True),
            nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(True),
            nn.Conv2d(w, w * 2, 3, 2, 1), nn.BatchNorm2d(w * 2), nn.ReLU(True),
            nn.Conv2d(w * 2, w * 2, 3, padding=1), nn.BatchNorm2d(w * 2), nn.ReLU(True),
        )
        dec = [nn.Upsample(scale_factor=2, mode="nearest"),
               nn.Conv2d(w * 2, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(True),
               nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(True)]
        head_ch = w
        if out_stride == 1:
            # Full resolution. The 3x3 peak suppression in decode_heatmap works in *heatmap*
            # pixels: at stride 2, two players 4px apart in the image are only 2 heatmap px
            # apart and one is suppressed as a duplicate.
            dec += [nn.Upsample(scale_factor=2, mode="nearest"),
                    nn.Conv2d(w, w // 2, 3, padding=1), nn.BatchNorm2d(w // 2), nn.ReLU(True)]
            head_ch = w // 2
        self.dec = nn.Sequential(*dec)
        self.head = nn.Conv2d(head_ch, n_classes, 1)
        self.head.bias.data.fill_(-2.19)      # start pessimistic: most pixels are background

    def forward(self, x):
        return self.head(self.dec(self.enc(x)))


def gaussian_heatmap(pos: np.ndarray, out_px: int, sigma: float, n_channels: int = 2
                     ) -> torch.Tensor:
    """positions (10, 2) -> (2, out_px, out_px). NaN or -1 = dead player, skipped."""
    xx, yy = _grid(out_px)
    hm = np.zeros((n_channels, out_px, out_px), np.float32)
    for p in range(pos.shape[0]):
        q = pos[p]
        if not np.isfinite(q).all() or q[0] < 0:
            continue
        ch = 0 if p < CFG.n_per_team else 1
        ix = int(np.clip(round(float(q[0]) * out_px), 0, out_px - 1))
        iy = int(np.clip(round(float(q[1]) * out_px), 0, out_px - 1))
        g = np.exp(-((xx - ix) ** 2 + (yy - iy) ** 2) / (2 * sigma ** 2))
        hm[ch] = np.maximum(hm[ch], g)
        hm[ch, iy, ix] = 1.0
    return torch.from_numpy(hm)


def focal_heatmap_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    p = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = target.ge(0.999).float()
    neg = 1.0 - pos
    pos_loss = -((1 - p) ** 2) * torch.log(p) * pos
    neg_loss = -((1 - target) ** 4) * (p ** 2) * torch.log(1 - p) * neg
    return (pos_loss.sum() + neg_loss.sum()) / pos.sum().clamp(min=1.0)


print(f"detector parameters: {sum(p.numel() for p in MinimapDetector().parameters()):,}")

### 10.1 Detector dataset — real labels when you have them

If `pilot_labels.json` exists it is used; otherwise synthetic frames stand in. Real labels come
from the classical finder plus your corrections (Section 12).

In [ ]:
def load_icon_labels(frames: np.ndarray, label_json: str, n_per_team: int = 5):
    """-> (frames_kept, positions (N, 10, 2)); slots 0-4 = T, 5-9 = CT, NaN where absent."""
    data = json.loads(Path(label_json).read_text())
    pts = {int(k): v for k, v in data["points"].items() if v}
    idx = sorted(pts)
    out = np.full((len(idx), n_per_team * 2, 2), np.nan, np.float32)
    for row, f in enumerate(idx):
        for ti in (0, 1):
            team = sorted([p[:2] for p in pts[f] if int(p[2]) == ti])[:n_per_team]
            if team:
                out[row, ti * n_per_team:ti * n_per_team + len(team)] = np.array(team, np.float32)
    return frames[idx], out


class IconDataset(Dataset):
    def __init__(self, frames: np.ndarray, positions: np.ndarray):
        self.frames, self.positions = frames, positions
        self.out_px = CFG.minimap_px // CFG.det_stride

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, i):
        x = PRE(self.frames[i])[0]
        y = gaussian_heatmap(self.positions[i], self.out_px, CFG.det_sigma)
        return x, y, torch.from_numpy(np.nan_to_num(self.positions[i], nan=-1.0))


def synthetic_icon_dataset(rounds: List[Round], n_frames: int, seed: int) -> IconDataset:
    rng = np.random.default_rng(seed)
    picks = [(r, t) for r in rounds for t in range(r.positions.shape[0])]
    sel = rng.permutation(len(picks))[:n_frames]
    fr = np.stack([picks[i][0].frames[picks[i][1]] for i in sel])
    po = np.stack([picks[i][0].positions[picks[i][1]] for i in sel])
    return IconDataset(fr, po)


USING_REAL_ICONS = Path(CFG.icon_labels_json).exists() and Path(CFG.frames_npy).exists()
if USING_REAL_ICONS:
    f_all, p_all = load_icon_labels(np.load(CFG.frames_npy), CFG.icon_labels_json)
    cut = int(len(f_all) * 0.8)
    det_train = IconDataset(f_all[:cut], p_all[:cut])
    det_val = IconDataset(f_all[cut:], p_all[cut:])
    print(f"REAL icon labels: {len(f_all)} frames -> train {len(det_train)} / val {len(det_val)}")
else:
    det_train = synthetic_icon_dataset(det_pool, 400, 10)
    det_val = synthetic_icon_dataset(det_pool_val, 120, 11)
    print(f"SYNTHETIC icons: train {len(det_train)} / val {len(det_val)}")

det_train_dl = DataLoader(det_train, batch_size=CFG.det_batch, shuffle=True, drop_last=True)
det_val_dl = DataLoader(det_val, batch_size=CFG.det_batch)

### 10.2 Decoding heatmaps back to coordinates

The 3×3 max-pool is cheap non-maximum suppression: keep a pixel only if it is the brightest in
its neighbourhood. Without it one player produces a cluster of adjacent "detections", like a
single voice echoing and being counted as a crowd.

In [ ]:
# Detection keeps a 2 px margin around the map: a correct peak can land a pixel past the edge
# (filtering at the exact edge cost 2.7 points of recall); a real off-map blob is far beyond it.
DETECT_MASK = cv2.dilate(MAP_MASK.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)


def _on_map(h: int, w: int) -> torch.Tensor:
    """DETECT_MASK at heatmap resolution, as a float tensor (1 on the map, 0 off it)."""
    m = DETECT_MASK if DETECT_MASK.shape == (h, w) else cv2.resize(
        DETECT_MASK.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return torch.from_numpy(m.astype(np.float32))


@torch.no_grad()
def decode_heatmap(logits: torch.Tensor, thresh: float, max_per_team: int):
    prob = torch.sigmoid(logits)
    peaks = (F.max_pool2d(prob, 3, 1, 1) == prob).float()
    B, C, h, w = prob.shape
    prob = prob * peaks * _on_map(h, w).to(prob.device)        # nothing off the map counts
    scores, idx = prob.view(B, C, -1).topk(max_per_team, dim=-1)
    ys = (idx // w).float() / h
    xs = (idx % w).float() / w
    out = []
    for b in range(B):
        d = {}
        for c, team in enumerate(CFG.teams[:C]):
            keep = scores[b, c] >= thresh
            d[team] = torch.stack([xs[b, c][keep], ys[b, c][keep],
                                   scores[b, c][keep]], -1).cpu().numpy()
        out.append(d)
    return out


@torch.no_grad()
def detector_metrics(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    tp = fp = fn = 0
    errs: List[float] = []
    for x, _, pos in loader:
        dets = decode_heatmap(model(x.to(DEVICE)), CFG.det_peak_thresh, CFG.n_per_team)
        for b in range(x.shape[0]):
            for c, team in enumerate(CFG.teams):
                gt = pos[b, c * CFG.n_per_team:(c + 1) * CFG.n_per_team].numpy()
                gt = gt[np.isfinite(gt[:, 0]) & (gt[:, 0] >= 0)]
                pred = dets[b][team]
                if len(gt) == 0:
                    fp += len(pred)
                    continue
                used: set = set()
                order = np.argsort(-pred[:, 2]) if len(pred) else []
                for pi in order:
                    d = np.linalg.norm(gt - pred[pi, :2], axis=1)
                    d[list(used)] = np.inf
                    j = int(np.argmin(d))
                    if d[j] <= CFG.det_match_radius:
                        tp += 1; used.add(j); errs.append(float(d[j]))
                    else:
                        fp += 1
                fn += len(gt) - len(used)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return {"precision": prec, "recall": rec,
            "f1": 2 * prec * rec / max(prec + rec, 1e-9),
            "mean_pos_err_norm": float(np.mean(errs)) if errs else float("nan")}

### 10.3 Train

Recall stays at zero for the first few epochs by design — the head starts biased towards
"background everywhere", and it takes a while before any peak clears the detection threshold.
Like a camera slowly pulling into focus: nothing, nothing, then an image.

In [ ]:
DET_HISTORY: List[Dict[str, float]] = []


def train_detector(epochs: int = None) -> MinimapDetector:
    epochs = CFG.det_epochs if epochs is None else epochs
    set_seed(CFG.seed)
    model = MinimapDetector().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.det_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    DET_HISTORY.clear()
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x, y, _ in det_train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = focal_heatmap_loss(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.shape[0]
        sched.step()
        m = detector_metrics(model, det_val_dl)
        DET_HISTORY.append({"epoch": ep + 1, "loss": tot / len(det_train), **m})
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            print(f"epoch {ep+1:3d}/{epochs}  loss {tot/len(det_train):.4f}  "
                  f"val P {m['precision']:.3f} R {m['recall']:.3f} F1 {m['f1']:.3f}  "
                  f"err {m['mean_pos_err_norm']*CFG.minimap_px:.2f}px")
    return model


def show_training_curve(history: List[Dict[str, float]]):
    ep = [h["epoch"] for h in history]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(ep, [h["loss"] for h in history], color="#33404d", lw=2)
    a1.set_title("training loss (lower is better)"); a1.set_xlabel("epoch"); a1.grid(alpha=0.3)
    a2.plot(ep, [h["recall"] * 100 for h in history], color="#d9822b", lw=2.2,
            label="recall: share of players found")
    a2.plot(ep, [h["precision"] * 100 for h in history], color="#2f7fd0", lw=2.2,
            label="precision: share of detections that are real")
    a2.set_ylim(0, 102); a2.set_ylabel("%"); a2.set_xlabel("epoch"); a2.grid(alpha=0.3)
    a2.set_title("validation precision and recall"); a2.legend(loc="lower right", fontsize=9)
    fig.suptitle(f"Model 1 training ({'real' if USING_REAL_ICONS else 'synthetic'} icon labels)")
    plt.tight_layout(); plt.show()


detector = train_detector()
show_training_curve(DET_HISTORY)
DET_METRICS = detector_metrics(detector, det_val_dl)
print("\nfinal detector:", {k: round(v, 4) for k, v in DET_METRICS.items()})
torch.save(detector.state_dict(), Path(CFG.out_dir) / "detector.pt")

### 10.4 What the detector sees

The same circled-icon view as Section 6, but drawn from the CNN rather than the colour rule —
so the two are directly comparable on the same frames.

In [ ]:
@torch.no_grad()
def show_detector_output(model, frames: np.ndarray, idxs=None, n: int = 5, show_heatmap=True):
    model.eval()
    idxs = idxs if idxs is not None else np.linspace(0, len(frames) - 1, n).astype(int)
    x = PRE(frames[idxs]).to(DEVICE)
    logits = model(x)
    prob = torch.sigmoid(logits).cpu().numpy()
    dets = decode_heatmap(logits, CFG.det_peak_thresh, CFG.n_per_team)
    rows = 2 if show_heatmap else 1
    fig, axes = plt.subplots(rows, len(idxs), figsize=(3.0 * len(idxs), 3.3 * rows), squeeze=False)
    for k, i in enumerate(idxs):
        ax = axes[0][k]
        ax.imshow(frames[i]); h, w = frames[i].shape[:2]
        nT = 0
        for c, team in enumerate(CFG.teams):
            for x_, y_, s_ in dets[k][team]:
                ax.plot(x_ * w, y_ * h, "o", mfc="none", mec=TEAM_EDGE[c], ms=13, mew=2)
                nT += (c == 0)
        ax.set_title(f"f{i} — {nT}T / {len(dets[k]['T'])+len(dets[k]['CT'])-nT}CT", fontsize=10)
        ax.axis("off")
        if show_heatmap:
            axes[1][k].imshow(prob[k].max(0), cmap="hot", vmin=0, vmax=1)
            axes[1][k].set_title("heatmap (max of both teams)", fontsize=8)
            axes[1][k].axis("off")
    fig.suptitle("Model 1 — CNN detections (orange = T, blue = CT)")
    plt.tight_layout(); plt.show()


_vis_frames = (np.load(CFG.frames_npy) if USING_REAL_ICONS else test_rounds[0].frames)
show_detector_output(detector, _vis_frames)

## 11. Tracking

The detector answers "where are the icons in this frame". The tracker answers "which icon is
the same player as last frame".

A textbook tracker discovers tracks: each unmatched detection starts a new one. On this data
that fails badly — because the detector misses some icons, every dropped frame orphans a track
and every recovery spawns a fresh one, so five players fragment into ~20 partial tracks and
"keep the longest five" grabs fragments instead of players. Measured on synthetic data, this
alone cost about 9 of the 10 pixels of total error.

**`FixedSlotTracker` uses the fact the free tracker throws away: there are exactly five players
per team.** Slots are created once and never destroyed; a slot that loses its detection coasts
along its last velocity instead of dying; a spare detection goes to whichever slot has been
lost longest.

The free tracker is a register that enrols a new pupil every time someone comes back from the
toilet. This one keeps five named seats and marks people temporarily absent.

In [ ]:
class FixedSlotTracker:
    def __init__(self, n_slots: int, gate: float, max_miss: int, smooth: float = 0.6):
        self.n_slots, self.gate, self.max_miss, self.smooth = n_slots, gate, max_miss, smooth

    def run(self, dets_per_frame: List[np.ndarray]) -> np.ndarray:
        n = len(dets_per_frame)
        out = np.full((n, self.n_slots, 2), np.nan, np.float32)
        pos: List[Optional[np.ndarray]] = [None] * self.n_slots
        vel = [np.zeros(2, np.float32) for _ in range(self.n_slots)]
        miss = [0] * self.n_slots

        for f, det in enumerate(dets_per_frame):
            pts = det[:, :2].astype(np.float32) if len(det) else np.zeros((0, 2), np.float32)
            active = [s for s in range(self.n_slots) if pos[s] is not None]
            pred = {s: pos[s] + vel[s] for s in active}          # constant-velocity guess
            pairs = sorted((float(np.linalg.norm(pts[d] - pred[s])), s, d)
                           for s in active for d in range(len(pts)))
            used_s, used_d = set(), set()
            for dist, s, d in pairs:
                if s in used_s or d in used_d or dist > self.gate:
                    continue
                new = pts[d]
                vel[s] = self.smooth * vel[s] + (1 - self.smooth) * (new - pos[s])
                pos[s], miss[s] = new, 0
                used_s.add(s); used_d.add(d)
            for d in (i for i in range(len(pts)) if i not in used_d):
                empty = [s for s in range(self.n_slots) if pos[s] is None]
                if empty:
                    s = empty[0]
                else:
                    lost = [s for s in range(self.n_slots) if s not in used_s and miss[s] > 0]
                    if not lost:
                        continue                                 # genuine false positive
                    s = max(lost, key=lambda k: miss[k])          # re-acquire the stalest slot
                pos[s], vel[s], miss[s] = pts[d], np.zeros(2, np.float32), 0
                used_s.add(s)
            for s in range(self.n_slots):
                if s in used_s or pos[s] is None:
                    continue
                miss[s] += 1
                if miss[s] <= self.max_miss:
                    pos[s] = np.clip(pos[s] + vel[s] * 0.5, 0.0, 1.0)   # coast, damped
                else:
                    vel[s] = np.zeros(2, np.float32)
            for s in range(self.n_slots):
                if pos[s] is not None:
                    out[f, s] = pos[s]

        for s in range(self.n_slots):                 # slots born late: carry first value back
            known = np.where(~np.isnan(out[:, s, 0]))[0]
            if len(known) == 0:
                out[:, s] = 0.5
            else:
                out[:known[0], s] = out[known[0], s]
        order = np.argsort(out[:, :, 0].mean(axis=0))              # canonical left-to-right
        return out[:, order]


@torch.no_grad()
def positions_from_video(model: nn.Module, frames: np.ndarray) -> np.ndarray:
    """Full vision front-end: pixels -> (T, 10, 2). This is the deployed path."""
    model.eval()
    x = PRE(frames).to(DEVICE)
    dets = decode_heatmap(model(x), CFG.det_peak_thresh, CFG.n_per_team)
    per_team = []
    for team in CFG.teams:
        tr = FixedSlotTracker(CFG.n_per_team, CFG.track_gate, CFG.track_max_miss)
        per_team.append(tr.run([d[team] for d in dets]))
    return np.concatenate(per_team, axis=1)


_est = positions_from_video(detector, test_rounds[0].frames)
check_contract("positions", _est, (CFG.seq_len, N_PLAYERS, 2))


def positional_error(est: np.ndarray, gt: np.ndarray) -> float:
    errs = []
    for t in range(est.shape[0]):
        for c in range(2):
            sl = slice(c * CFG.n_per_team, (c + 1) * CFG.n_per_team)
            d = np.linalg.norm(est[t, sl][:, None, :] - gt[t, sl][None, :, :], axis=-1)
            errs.extend(d.min(axis=1).tolist())
    return float(np.mean(errs))


errs = [positional_error(positions_from_video(detector, r.frames), r.positions)
        for r in test_rounds[:20]]
print(f"detector alone : {DET_METRICS['mean_pos_err_norm']*CFG.minimap_px:5.2f} px")
print(f"+ tracker      : {np.mean(errs)*CFG.minimap_px:5.2f} px")
print("The detector is sub-pixel accurate; error far above that is the tracker's doing,\n"
      "which is exactly why the two are measured separately.")

## 12. Correcting labels by hand

The classical finder proposes, you correct. Reviewing is roughly five times faster than
clicking from scratch, because most frames need no change — you glance and press `n`.

**Team is read from the pixel you click**, so there are no key presses for orange versus blue.

```
  left click   add an icon        n / b   next / previous frame
  right click  remove the nearest s / q   save / save and quit
```

This needs an interactive window, so run it from a terminal or with `%matplotlib widget` —
`%matplotlib inline` renders static images and the clicks go nowhere.

**One rule while reviewing:** when icons overlap at spawn, click all five anyway and estimate
the centres. The classical finder reports one blob there and cannot do better; the CNN *can*
learn to place several peaks inside one blob, but only if your labels show it that. Label a
cluster as one player and you teach it to reproduce the threshold's own failure.

In [ ]:
def auto_label_icons(frames: np.ndarray, out_json: str, team_hues=None, background=None,
                     preview: int = 6) -> Dict:
    points = {i: find_icons_teams(f, team_hues, background) for i, f in enumerate(frames)}
    nT = np.array([sum(p[2] == 0 for p in v) for v in points.values()])
    nC = np.array([sum(p[2] == 1 for p in v) for v in points.values()])
    Path(out_json).write_text(json.dumps(
        {"schema": "xy_team", "n_frames": int(len(frames)),
         "points": {str(k): v for k, v in points.items() if v}}, indent=1))
    print(f"wrote {out_json}")
    print(f"  T  mean {nT.mean():.2f}/frame, {int((nT == 5).sum())} frames with 5")
    print(f"  CT mean {nC.mean():.2f}/frame, {int((nC == 5).sum())} frames with 5")
    if preview:
        show_detections(frames, np.linspace(0, len(frames) - 1, preview).astype(int),
                        team_hues=team_hues, background=background,
                        title="auto-proposed labels — correct these in ReviewTool")
    return points


class ReviewTool:
    """Correct machine-proposed icon labels. See the controls above."""

    def __init__(self, frames: np.ndarray, out_json: str, team_hues=None):
        self.frames, self.out = frames, Path(out_json)
        self.hues = team_hues or DEFAULT_TEAM_HUES
        self.labels: Dict[int, List[List[float]]] = {}
        if self.out.exists():
            d = json.loads(self.out.read_text())
            self.labels = {int(k): v for k, v in d.get("points", {}).items()}
            print(f"resuming — {len(self.labels)} frames already labelled")
        self.i = 0

    def _team_at(self, fx, fy) -> int:
        f = self.frames[self.i]; h, w = f.shape[:2]
        px, py = int(fx * w), int(fy * h)
        patch = f[max(py - 2, 0):py + 3, max(px - 2, 0):px + 3].reshape(1, -1, 3)
        hsv = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)[0]
        sat = hsv[:, 1].astype(float)
        hue = float(np.median(hsv[sat >= np.percentile(sat, 60), 0]))
        return CFG.teams.index(min(CFG.teams, key=lambda t: _hue_dist(hue, self.hues[t])))

    def _draw(self):
        self.ax.clear(); self.ax.imshow(self.frames[self.i])
        h, w = self.frames[self.i].shape[:2]
        pts = self.labels.get(self.i, [])
        for x, y, t in pts:
            self.ax.plot(x * w, y * h, "o", mfc="none", mec=TEAM_EDGE[int(t)], ms=14, mew=2)
        nT = sum(int(p[2]) == 0 for p in pts)
        self.ax.set_title(f"frame {self.i+1}/{len(self.frames)} — {nT} T, {len(pts)-nT} CT")
        self.ax.axis("off"); self.fig.canvas.draw_idle()

    def _click(self, ev):
        if ev.inaxes != self.ax or ev.xdata is None:
            return
        h, w = self.frames[self.i].shape[:2]
        cx, cy = float(ev.xdata) / w, float(ev.ydata) / h
        pts = self.labels.setdefault(self.i, [])
        if ev.button == 1:
            pts.append([cx, cy, float(self._team_at(cx, cy))])
        elif ev.button == 3 and pts:
            pts.pop(int(np.argmin([(p[0] - cx) ** 2 + (p[1] - cy) ** 2 for p in pts])))
        self._draw()

    def _key(self, ev):
        if ev.key in ("n", "enter", " "):
            self.i = min(self.i + 1, len(self.frames) - 1)
        elif ev.key == "b":
            self.i = max(self.i - 1, 0)
        elif ev.key == "s":
            self.save()
        elif ev.key == "q":
            self.save(); plt.close(self.fig); return
        self._draw()

    def save(self):
        self.out.write_text(json.dumps(
            {"schema": "xy_team", "n_frames": int(len(self.frames)),
             "points": {str(k): v for k, v in self.labels.items() if v}}, indent=1))
        print(f"saved {len(self.labels)} frames -> {self.out}")

    def run(self):
        self.fig, self.ax = plt.subplots(figsize=(8, 8))
        self.fig.canvas.mpl_connect("button_press_event", self._click)
        self.fig.canvas.mpl_connect("key_press_event", self._key)
        self._draw(); plt.show()


print("to label real footage:")
print("  auto_label_icons(pilot_frames, CFG.icon_labels_json, HUES, BG)")
print("  ReviewTool(pilot_frames, CFG.icon_labels_json, HUES).run()   # needs a live window")

## 13. Trajectory representation

The classifier cannot eat a list of dots; it needs a fixed-size tensor. Two options, both
implemented, because they are worth comparing:

**`coords` — `(32, 20)`.** Every player's x and y at every step. Keeps everything, but treats
slot 2 as a meaningful identity when our slots are arbitrary.

**`features` — `(32, 16)`.** Per team per step: centroid, spread, distance to A, distance to B,
fraction near each site, mean speed. Immune to the ordering problem, because swapping two
players does not move a centroid.

`coords` is a class register with every pupil's seat; `features` is "24 pupils, bunched near
the front, drifting left". For *which way did the class move*, the summary is usually enough
and cannot be confused by two pupils swapping seats.

In [ ]:
def to_features(pos: np.ndarray) -> np.ndarray:
    vel = np.diff(pos, axis=0, prepend=pos[:1])
    feats = []
    for c in range(2):
        sl = slice(c * CFG.n_per_team, (c + 1) * CFG.n_per_team)
        team, tvel = pos[:, sl], vel[:, sl]
        centroid = team.mean(axis=1)
        spread = np.linalg.norm(team - centroid[:, None, :], axis=-1).mean(axis=1)
        d_a = np.linalg.norm(centroid - A_SITE[None], axis=-1)
        d_b = np.linalg.norm(centroid - B_SITE[None], axis=-1)
        near_a = (np.linalg.norm(team - A_SITE[None, None], axis=-1) < 0.18).mean(axis=1)
        near_b = (np.linalg.norm(team - B_SITE[None, None], axis=-1) < 0.18).mean(axis=1)
        speed = np.linalg.norm(tvel, axis=-1).mean(axis=1)
        feats += [centroid[:, 0], centroid[:, 1], spread, d_a, d_b, near_a, near_b, speed]
    return np.stack(feats, axis=-1).astype(np.float32)


def to_sequence(pos: np.ndarray, mode: str = "features") -> np.ndarray:
    if mode == "features":
        return to_features(pos)
    if mode == "coords":
        return pos.reshape(pos.shape[0], -1).astype(np.float32)
    if mode == "both":
        return np.concatenate([to_features(pos), pos.reshape(pos.shape[0], -1)], -1).astype(np.float32)
    raise ValueError(mode)


FEATURE_NAMES = [f"{t}_{n}" for t in CFG.teams
                 for n in ["cx", "cy", "spread", "dist_A", "dist_B", "near_A", "near_B", "speed"]]
check_contract("sequence", to_sequence(train_rounds[0].positions), (CFG.seq_len, 16))

### 13.1 Do the classes separate at all?

The cheapest sanity check in the project, and it costs one plot. If attacker distance-to-A does
not visibly differ between `A_RUSH` and `B_RUSH`, no architecture will save you. Rushes should
also fall *faster* than splits — that speed difference is what separates the two.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, (fi, name) in zip(axes, [(3, "T distance to A site"), (2, "T spread"), (7, "T mean speed")]):
    for c in T_CLASSES:
        curves = np.stack([to_features(r.positions)[:, fi] for r in train_rounds if r.label == c])
        ax.plot(curves.mean(0), lw=2, label=c)
    ax.set_title(name); ax.set_xlabel("timestep (before the plant)")
axes[0].legend(fontsize=8)
fig.suptitle("Question 1 - attacker tactics, before the plant")
plt.tight_layout(); plt.show()


def ct_post_signals(r: Round):
    """Defender signals after the plant, measured against the planted site."""
    site = A_SITE if r.plant_site == "A" else B_SITE
    d = np.linalg.norm(r.post_positions[:, CFG.n_per_team:] - site[None, None], axis=-1)
    return d.mean(1), (d < 0.10).sum(1), d.min(1)


fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
titles = ["CT mean distance to the planted site", "CTs standing on the site", "closest CT to the site"]
for c in CT_CLASSES:
    sig = [ct_post_signals(r) for r in train_rounds if r.ct_label == c]
    for k, ax in enumerate(axes):
        ax.plot(np.stack([s[k] for s in sig]).mean(0), lw=2, label=c)
for ax, t in zip(axes, titles):
    ax.set_title(t); ax.set_xlabel("timestep (after the plant)")
axes[0].legend(fontsize=8)
fig.suptitle("Question 2 - defender responses, after the plant")
plt.tight_layout(); plt.show()

## 14. Model 2 — the tactic classifier

Three models, deliberately from three different families:

| Name | Type | Idea |
|---|---|---|
| `feature_mlp` | MLP (feedforward) | pools over time — **order-blind on purpose** |
| `gru` | RNN (bidirectional GRU) | reads the sequence in order, keeping a running memory |
| `tcn` | 1-D CNN over time | slides filters along time; tactical patterns are *local* |
| `transformer` | self-attention | sees all steps at once, learns which moments matter |

The MLP is the baseline that must be beaten, not an afterthought. It throws time order away
entirely, so if a temporal model cannot beat it, the temporal information is not being used.

The `tcn` is the one I would bet on here: far fewer parameters than the transformer, and a
split or a rush happens over a few seconds rather than across the whole round, so locality is
the right assumption when labelled rounds are scarce.

In [ ]:
class FeatureMLP(nn.Module):
    def __init__(self, d_in, hidden, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in * 3, hidden), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.ReLU(True), nn.Linear(hidden, n_classes))

    def forward(self, x):
        return self.net(torch.cat([x.mean(1), x.amax(1), x[:, -1]], -1))


class GRUClassifier(nn.Module):
    def __init__(self, d_in, hidden, n_classes, layers=1):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.gru = nn.GRU(d_in, hidden, layers, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden * 2, n_classes))

    def forward(self, x):
        out, _ = self.gru(self.norm(x))
        return self.head(out.mean(1))


class TCNClassifier(nn.Module):
    """Dilated 1-D convolutions over time: cheap, local, and well suited to small datasets."""

    def __init__(self, d_in, hidden, n_classes):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.net = nn.Sequential(
            nn.Conv1d(d_in, hidden, 3, padding=1, dilation=1), nn.BatchNorm1d(hidden), nn.ReLU(True),
            nn.Conv1d(hidden, hidden, 3, padding=2, dilation=2), nn.BatchNorm1d(hidden), nn.ReLU(True),
            nn.Conv1d(hidden, hidden, 3, padding=4, dilation=4), nn.BatchNorm1d(hidden), nn.ReLU(True))
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden * 2, n_classes))

    def forward(self, x):
        h = self.net(self.norm(x).transpose(1, 2))
        return self.head(torch.cat([h.mean(-1), h.amax(-1)], -1))


class TinyTransformer(nn.Module):
    def __init__(self, d_in, hidden, n_classes, heads=4, layers=2):
        super().__init__()
        self.proj = nn.Linear(d_in, hidden)
        self.pos = nn.Parameter(torch.randn(1, CFG.seq_len + 1, hidden) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, hidden) * 0.02)
        enc = nn.TransformerEncoderLayer(hidden, heads, hidden * 2, dropout=0.1,
                                         batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        h = self.proj(x)
        h = torch.cat([self.cls.expand(h.shape[0], -1, -1), h], 1)
        return self.head(self.enc(h + self.pos[:, :h.shape[1]])[:, 0])


MODEL_ZOO = {"feature_mlp": FeatureMLP, "gru": GRUClassifier,
             "tcn": TCNClassifier, "transformer": TinyTransformer}
for k, v in MODEL_ZOO.items():
    n_t = sum(p.numel() for p in v(16, CFG.cls_hidden, len(T_CLASSES)).parameters())
    n_ct = sum(p.numel() for p in v(16, CFG.cls_hidden, len(CT_CLASSES)).parameters())
    print(f"{k:<13} {n_t:>8,} params (attacker head)   {n_ct:>8,} (defender head)")

### 14.1 Dataset, metrics, training

**Two questions, same four models.** Question 1 reads the window before the plant and picks an
attacker tactic for every round. Question 2 reads the window after the plant and picks a
defender response, only for rounds where the bomb went down. Each question gets its own
dataset, normaliser and best model.

**Macro F1, not accuracy.** If 40% of rounds are `A_RUSH`, a model that always guesses
`A_RUSH` scores 40% accuracy while being useless. Macro F1 averages per class, so ignoring a
rare class such as `SAVE` is punished.

In [ ]:
class SequenceDataset(Dataset):
    """One question over a set of rounds. phase="pre": attacker tactic, every round.
    phase="post": defender response, only rounds where the bomb was planted."""

    def __init__(self, rounds: List[Round], mode="features", source="gt",
                 detector=None, normaliser=None, phase: str = "pre"):
        self.phase, self.classes = phase, PHASES[phase]
        self.n_classes = len(self.classes)
        X, y = [], []
        for r in rounds:
            lab = r.label_of(phase)
            if lab is None:
                continue
            pos = (r.positions_of(phase) if source == "gt"
                   else positions_from_video(detector, r.frames_of(phase)))
            X.append(to_sequence(pos, mode)); y.append(self.classes.index(lab))
        self.X = torch.from_numpy(np.stack(X))
        self.y = torch.tensor(y, dtype=torch.long)
        if normaliser is None:
            self.mu = self.X.mean((0, 1), keepdim=True)
            self.sd = self.X.std((0, 1), keepdim=True).clamp(min=1e-6)
        else:
            self.mu, self.sd = normaliser
        self.X = (self.X - self.mu) / self.sd

    @property
    def normaliser(self):
        return (self.mu, self.sd)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def confusion_matrix(y_true, y_pred, n):
    cm = np.zeros((n, n), int)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        cm[t, p] += 1
    return cm


def macro_f1(cm):
    f1s = []
    for c in range(cm.shape[0]):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
        pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        f1s.append(2 * pr * rc / max(pr + rc, 1e-9))
    return float(np.mean(f1s)), np.array(f1s)


@torch.no_grad()
def evaluate(model, ds):
    model.eval()
    pred = model(ds.X.to(DEVICE)).cpu().argmax(-1)
    cm = confusion_matrix(ds.y, pred, ds.n_classes)
    mf1, per = macro_f1(cm)
    return {"accuracy": float((pred == ds.y).float().mean()), "macro_f1": mf1,
            "per_class_f1": per, "confusion": cm}


def train_classifier(name, train_ds, val_ds, verbose=True):
    set_seed(CFG.seed)
    model = MODEL_ZOO[name](train_ds.X.shape[-1], CFG.cls_hidden, train_ds.n_classes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.cls_lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.cls_epochs)
    dl = DataLoader(train_ds, batch_size=CFG.cls_batch, shuffle=True)
    best_f1, best_state = -1.0, None
    for ep in range(CFG.cls_epochs):
        model.train()
        for xb, yb in dl:
            loss = F.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        m = evaluate(model, val_ds)
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose and (ep + 1) % 20 == 0:
            print(f"  epoch {ep+1:3d}  val acc {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}")
    model.load_state_dict(best_state)
    return model, evaluate(model, val_ds)

In [ ]:
REPR = "features"
DATA: Dict[str, Dict[str, SequenceDataset]] = {}
for phase in ("pre", "post"):
    tr = SequenceDataset(train_rounds, REPR, "gt", phase=phase)
    DATA[phase] = {"train": tr,
                   "val": SequenceDataset(val_rounds, REPR, "gt", normaliser=tr.normaliser, phase=phase),
                   "test": SequenceDataset(test_rounds, REPR, "gt", normaliser=tr.normaliser, phase=phase)}
train_ds, val_ds, test_ds = (DATA["pre"][k] for k in ("train", "val", "test"))

RESULTS: Dict[str, Dict] = {}
for phase, title in (("pre", "QUESTION 1 - attacker tactic, before the plant"),
                     ("post", "QUESTION 2 - defender response, after the plant")):
    d = DATA[phase]
    print(f"\n{title}   (train {len(d['train'])} / val {len(d['val'])} / test {len(d['test'])} rounds)")
    RESULTS[phase] = {}
    for name in MODEL_ZOO:
        model, vm = train_classifier(name, d["train"], d["val"], verbose=False)
        tm = evaluate(model, d["test"])
        RESULTS[phase][name] = {"model": model, "val": vm, "test": tm}
        print(f"  {name:<12} val macro-F1 {vm['macro_f1']:.3f}   "
              f"TEST acc {tm['accuracy']:.3f}  macro-F1 {tm['macro_f1']:.3f}")

BEST = {ph: max(RESULTS[ph], key=lambda k: RESULTS[ph][k]["val"]["macro_f1"]) for ph in RESULTS}
results, best_name = RESULTS["pre"], BEST["pre"]
best_model = results[best_name]["model"]
ct_best_name = BEST["post"]
ct_model = RESULTS["post"][ct_best_name]["model"]
print(f"\nbest on validation: attacker question -> {best_name} | defender question -> {ct_best_name}")

In [ ]:
def plot_confusion(cm, title, classes: Sequence[str] = CLASSES):
    n = len(classes)
    fig, ax = plt.subplots(figsize=(5.2, 4.4) if n > 3 else (4.2, 3.7))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()


plot_confusion(results[best_name]["test"]["confusion"],
               f"Q1 attacker tactic - {best_name} - oracle positions (test)")
for c, f1 in zip(T_CLASSES, results[best_name]["test"]["per_class_f1"]):
    print(f"  {c:<9} F1 {f1:.3f}")
ct_test = RESULTS["post"][ct_best_name]["test"]
plot_confusion(ct_test["confusion"],
               f"Q2 defender response - {ct_best_name} - oracle positions (test)", CT_CLASSES)
for c, f1 in zip(CT_CLASSES, ct_test["per_class_f1"]):
    print(f"  {c:<9} F1 {f1:.3f}")

## 15. Experiment A vs B — the headline result

* **A (oracle):** exact positions → trajectory → classifier.
* **B (vision):** radar pixels → detector → tracker → trajectory → **the same classifier
  weights**.

The gap is the price of seeing pixels instead of knowing coordinates. The classifier is *not*
retrained for B, which keeps the comparison honest: the only thing that changed is input
quality.

In [ ]:
vision_test_ds = SequenceDataset(test_rounds, REPR, "vision", detector=detector,
                                 normaliser=train_ds.normaliser)
exp_a, exp_b = evaluate(best_model, test_ds), evaluate(best_model, vision_test_ds)
ct_vision_test = SequenceDataset(test_rounds, REPR, "vision", detector=detector,
                                 normaliser=DATA["post"]["train"].normaliser, phase="post")
ct_a, ct_b = evaluate(ct_model, DATA["post"]["test"]), evaluate(ct_model, ct_vision_test)
print(f"{'':<34}{'accuracy':>10}{'macro-F1':>10}")
for q, a, b in (("Q1 attacker", exp_a, exp_b), ("Q2 defender", ct_a, ct_b)):
    print(f"{q + ' A - oracle positions':<34}{a['accuracy']:>10.3f}{a['macro_f1']:>10.3f}")
    print(f"{q + ' B - vision pipeline':<34}{b['accuracy']:>10.3f}{b['macro_f1']:>10.3f}")
    print(f"{q + ' gap':<34}{a['accuracy']-b['accuracy']:>10.3f}{a['macro_f1']-b['macro_f1']:>10.3f}")
plot_confusion(exp_b["confusion"], f"Q1 attacker tactic - {best_name} - full vision pipeline (test)")

# --- B': retrain on vision trajectories -------------------------------------------------
# B alone conflates two different things: information genuinely lost by the camera, and the
# classifier simply never having seen tracker noise. B' separates them by training on vision
# trajectories too. If B' climbs back to A, the gap was distribution shift and the fix is free;
# if B' stays below A, the information is really gone, and that is the finding.
RUN_BPRIME = True
if RUN_BPRIME:
    rng_bp = np.random.default_rng(CFG.seed + 9)
    bg_bp = build_background(CFG.minimap_px, np.random.default_rng(1234), MAP_BG_SOURCE)
    bp_rounds = train_rounds[:60]
    for r in bp_rounds:                       # render only the subset B' needs
        if r.frames is None:
            r.frames = render_frames(r.positions, bg_bp, CFG.minimap_px, rng_bp)
    bp_train = SequenceDataset(bp_rounds, REPR, "vision", detector=detector)
    bp_val = SequenceDataset(test_rounds[:20], REPR, "vision", detector=detector,
                             normaliser=bp_train.normaliser)
    bp_model, _ = train_classifier(best_name, bp_train, bp_val, verbose=False)
    bp_test = SequenceDataset(test_rounds, REPR, "vision", detector=detector,
                              normaliser=bp_train.normaliser)
    exp_bp = evaluate(bp_model, bp_test)
    print(f"{chr(66)+chr(39)+' - trained on vision':<24}{exp_bp['accuracy']:>10.3f}"
          f"{exp_bp['macro_f1']:>10.3f}   (n_train={len(bp_rounds)})")
else:
    exp_bp = None

## 16. Labelling rounds from a demo recording

Icon labels train Model 1; **round labels train Model 2**, and they are the real bottleneck —
roughly 200 labelled rounds, which is 8–10 full matches.

```
  left / right   scrub (shift = 10x)        1-5    attacker tactic (A_RUSH B_RUSH A_SPLIT B_SPLIT DEFAULT)
  [ / ]          mark round start / end     a / b  bomb planted now, at A / B
  6 / 7 / 8      defender response          x      clear the plant
                 (RETAKE TRICKLE SAVE)      t / c  winner: T or CT
  u              undo last round            enter  commit the round
  s / q          save / save and quit
```

Mark the plant at the moment the bomb goes down. Everything before it is the attackers'
window; everything after it is the defenders' window.

**The time-window problem.** During training, round boundaries and the plant come from you. At
deployment there is no annotator, so pixels alone must decide where a round starts and when the
bomb went down: a sliding window (Section 17), reading the round timer, spotting the planted
bomb on the radar, or an analyst marking rounds. Pick one deliberately — this is the one place
a `.dem` dependency can sneak back in unnoticed.

In [ ]:
def rounds_json_for(video) -> Path:
    """labels/<match_id>.rounds.json - one label file per replay in data/."""
    return Path(CFG.labels_dir) / f"{match_id_of(video)}.rounds.json"


class RoundLabeller:
    """Keys: left/right scrub (shift = 10x) | [ ] round start / end | 1-5 attacker tactic |
    a / b bomb planted now at A / B | 6 7 8 defender response RETAKE TRICKLE SAVE | x clear plant |
    t / c winner | enter commit | u undo | s save | q save and quit."""

    def __init__(self, video: str, crop_box, out_json: str = None, step_s: float = 2.0):
        assert cv2 is not None, "pip install opencv-python"
        self.cap = cv2.VideoCapture(video)
        if not self.cap.isOpened():
            raise FileNotFoundError(video)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 60.0
        self.dur = self.cap.get(cv2.CAP_PROP_FRAME_COUNT) / self.fps
        self.box, self.step = crop_box, step_s
        self.out = Path(out_json or rounds_json_for(video))
        self.out.parent.mkdir(parents=True, exist_ok=True)
        self.t = 0.0
        self._reset()
        self.rounds: List[Dict] = []
        if self.out.exists():
            self.rounds = json.loads(self.out.read_text())["rounds"]
            print(f"resuming - {len(self.rounds)} rounds already marked")
        print(f"video {self.dur:.0f}s @ {self.fps:.0f}fps")

    def _reset(self):
        self.start = self.end = self.label = self.winner = None
        self.plant = self.site = self.ct_label = None

    def _frame(self):
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, int(self.t * self.fps))
        ok, bgr = self.cap.read()
        if not ok:
            return np.zeros((100, 100, 3), np.uint8)
        x0, y0, x1, y1 = self.box
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[y0:y1, x0:x1]

    def _draw(self):
        self.ax.clear(); self.ax.imshow(self._frame())
        f = lambda v: "-" if v is None else f"{v:.0f}s"
        plant = "-" if self.plant is None else f"{self.site} at {self.plant:.0f}s"
        self.ax.set_title(f"t={self.t:.0f}s / {self.dur:.0f}s   round {len(self.rounds)+1}\n"
                          f"start {f(self.start)}  end {f(self.end)}  attackers {self.label or '-'}\n"
                          f"plant {plant}  defenders {self.ct_label or '-'}  "
                          f"winner {self.winner or '-'}", fontsize=10)
        self.ax.axis("off"); self.fig.canvas.draw_idle()

    def _commit(self):
        if self.start is None or self.end is None or not self.label:
            print("need start ([), end (]) and an attacker tactic (1-5)"); return
        if self.end <= self.start:
            print("end must come after start"); return
        if self.plant is not None and not (self.start < self.plant < self.end):
            print("the plant must fall between start and end"); return
        if self.ct_label and self.plant is None:
            print("a defender response (6-8) needs a plant (a / b)"); return
        if self.plant is not None and not self.ct_label:
            print("note: plant marked but no defender response (6-8) - saved without one")
        self.rounds.append({"round_idx": len(self.rounds), "start_s": round(self.start, 2),
                            "end_s": round(self.end, 2), "label": self.label,
                            "winner": self.winner,
                            "plant_s": None if self.plant is None else round(self.plant, 2),
                            "plant_site": self.site, "ct_label": self.ct_label})
        print(f"round {len(self.rounds)}: {self.label}  plant {self.site or '-'}  "
              f"defenders {self.ct_label or '-'}  winner {self.winner or '?'}")
        self._reset()

    def _key(self, ev):
        step = self.step * (10 if ev.key in ("shift+right", "shift+left") else 1)
        if ev.key in ("right", "shift+right"):
            self.t = min(self.t + step, self.dur - 1)
        elif ev.key in ("left", "shift+left"):
            self.t = max(self.t - step, 0.0)
        elif ev.key == "[":
            self.start = self.t
        elif ev.key == "]":
            self.end = self.t
        elif ev.key in list("12345"):
            self.label = T_CLASSES[int(ev.key) - 1]
        elif ev.key in ("a", "b"):
            self.plant, self.site = self.t, ev.key.upper()
        elif ev.key in ("6", "7", "8"):
            self.ct_label = CT_CLASSES[int(ev.key) - 6]
        elif ev.key == "x":
            self.plant = self.site = self.ct_label = None
        elif ev.key in ("t", "c"):
            self.winner = "T" if ev.key == "t" else "CT"
        elif ev.key == "enter":
            self._commit()
        elif ev.key == "u" and self.rounds:
            print("removed", self.rounds.pop()["label"])
        elif ev.key == "s":
            self.save()
        elif ev.key == "q":
            self.save(); plt.close(self.fig); return
        self._draw()

    def save(self):
        self.out.write_text(json.dumps({"attacker_tactics": T_CLASSES, "defender_responses": CT_CLASSES,
                                        "rounds": self.rounds}, indent=1))
        print(f"saved {len(self.rounds)} rounds -> {self.out}   "
              f"{ {t: sum(r['label'] == t for r in self.rounds) for t in T_CLASSES} }   "
              f"{ {t: sum(r.get('ct_label') == t for r in self.rounds) for t in CT_CLASSES} }")

    def run(self):
        for k in [k for k in plt.rcParams if k.startswith("keymap.")]:
            plt.rcParams[k] = []      # matplotlib's own shortcuts (s, c, q, arrows) would also fire
        self.fig, self.ax = plt.subplots(figsize=(7, 7.8))
        self.fig.canvas.mpl_connect("key_press_event", self._key)
        self._draw(); plt.show()


def rounds_to_clips(video: str, crop_box, rounds_json: str = None) -> List[Dict]:
    """One video's marked rounds -> dicts with the pre-plant clip and, if planted, the post-plant clip."""
    rounds = json.loads(Path(rounds_json or rounds_json_for(video)).read_text())["rounds"]
    out = []
    for r in rounds:
        try:
            pre_end = r.get("plant_s") or r["end_s"]
            pre = extract_clip(str(video), r["start_s"], pre_end, crop_box, CFG.seq_len)
            post = (extract_clip(str(video), r["plant_s"], r["end_s"], crop_box, CFG.seq_len)
                    if r.get("plant_s") else None)
            out.append({"pre": pre, "post": post, "label": r["label"], "winner": r.get("winner") or "",
                        "round_idx": r["round_idx"], "plant_site": r.get("plant_site") or "",
                        "ct_label": r.get("ct_label") or ""})
        except RuntimeError as e:
            print(f"skipped {match_id_of(video)} round {r['round_idx']}: {e}")
    return out


def build_rounds_dataset(videos: Sequence[Path] = None, out_npz: str = None) -> None:
    """Every labelled video in data/ -> one npz, tagged with match_id so splits stay by match."""
    rows, match_ids = [], []
    for v in (VIDEOS if videos is None else videos):
        if not rounds_json_for(v).exists():
            print(f"{v.name:<40} no labels yet ({rounds_json_for(v)}) - skipped")
            continue
        rs = rounds_to_clips(v, CROPS[v.name])
        print(f"{v.name:<40} {len(rs)} rounds, {sum(r['post'] is not None for r in rs)} planted")
        rows += rs; match_ids += [match_id_of(v)] * len(rs)
    if not rows:
        print("no labelled rounds found - run RoundLabeller first"); return
    pre = stack_resized([r["pre"][None] for r in rows])                 # (R, seq_len, H, W, 3)
    post = np.zeros_like(pre)
    for k, r in enumerate(rows):
        if r["post"] is not None:
            post[k] = stack_resized([pre[k][None], r["post"][None]])[1]
    np.savez_compressed(out_npz or CFG.rounds_npz, frames=pre, post_frames=post,
                        has_post=np.array([r["post"] is not None for r in rows]),
                        labels=np.array([r["label"] for r in rows]),
                        winners=np.array([r["winner"] for r in rows]),
                        round_idx=np.array([r["round_idx"] for r in rows]),
                        plant_site=np.array([r["plant_site"] for r in rows]),
                        ct_labels=np.array([r["ct_label"] for r in rows]),
                        match_ids=np.array(match_ids))
    print(f"{len(rows)} rounds from {len(set(match_ids))} match(es) -> {out_npz or CFG.rounds_npz}")


def load_real_rounds(npz_path: str = None) -> List[Round]:
    d = np.load(npz_path or CFG.rounds_npz, allow_pickle=True)
    n = len(d["labels"])
    get = lambda k, default: d[k] if k in d else [default] * n
    out = []
    for k in range(n):
        has_post = bool(get("has_post", False)[k])
        out.append(Round(str(get("match_ids", "match_000")[k]), int(get("round_idx", k)[k]),
                         str(d["labels"][k]), None, d["frames"][k],
                         str(get("winners", "")[k]) or None,
                         str(get("plant_site", "")[k]) or None, str(get("ct_labels", "")[k]) or None,
                         None, d["post_frames"][k] if has_post else None))
    return out


print("label one match:   RoundLabeller(str(VIDEOS[k]), CROPS[VIDEOS[k].name]).run()")
print("                   -> saves", Path(CFG.labels_dir) / "<match_id>.rounds.json")
print("then:              build_rounds_dataset()   # every labelled video in data/ -> one npz")
print("then:              real_rounds = load_real_rounds()   # Round.match_id = video file name")

## 17. End-to-end inference and scouting output

Pixels in, tactical predictions out. No `.dem` anywhere in this path.

In [ ]:
class StratScoutPipeline:
    """Pixels in, two answers out. heads: phase -> (classifier, normaliser)."""

    def __init__(self, detector, heads: Dict[str, Tuple[nn.Module, Tuple]], repr_mode=REPR):
        self.detector = detector.eval()
        self.heads = {ph: (m.eval(), norm) for ph, (m, norm) in heads.items()}
        self.repr_mode = repr_mode

    @torch.no_grad()
    def predict_window(self, frames: np.ndarray, phase: str = "pre") -> Dict:
        model, (mu, sd) = self.heads[phase]
        classes = PHASES[phase]
        pos = positions_from_video(self.detector, frames)
        seq = (torch.from_numpy(to_sequence(pos, self.repr_mode))[None] - mu) / sd
        probs = F.softmax(model(seq.to(DEVICE)), -1)[0].cpu().numpy()
        k = int(probs.argmax())
        return {"label": classes[k], "confidence": float(probs[k]), "positions": pos,
                "probabilities": {c: float(p) for c, p in zip(classes, probs)}}

    @torch.no_grad()
    def predict_video(self, frames: np.ndarray, stride: int = None, min_confidence: float = 0.0):
        w = CFG.seq_len
        stride = stride or w // 2
        out = []
        for s in range(0, max(len(frames) - w + 1, 1), stride):
            win = frames[s:s + w]
            if len(win) < w:
                break
            r = self.predict_window(win)
            if r["confidence"] >= min_confidence:
                out.append({"start_time": s / CFG.fps_sampled,
                            "end_time": (s + w) / CFG.fps_sampled,
                            "label": r["label"], "confidence": r["confidence"]})
        return out


pipeline = StratScoutPipeline(detector, {"pre": (best_model, DATA["pre"]["train"].normaliser),
                                         "post": (ct_model, DATA["post"]["train"].normaliser)})
demo_round = next(r for r in test_rounds if r.planted)
one = pipeline.predict_window(demo_round.frames, "pre")
two = pipeline.predict_window(demo_round.post_frames, "post")
print(f"before the plant: true {demo_round.label:<8} -> predicted {one['label']} ({one['confidence']:.2f})")
print(f"after the plant at {demo_round.plant_site}: true {demo_round.ct_label:<8} -> "
      f"predicted {two['label']} ({two['confidence']:.2f})")

In [ ]:
def scouting_report(pipeline, rounds: List[Round], min_confidence: float = 0.0) -> Dict:
    rows = []
    for r in rounds:
        pre = pipeline.predict_window(r.frames, "pre")
        row = {"match_id": r.match_id, "round": r.round_idx, "true": r.label,
               "label": pre["label"], "confidence": pre["confidence"],
               "site": r.plant_site, "ct_true": r.ct_label, "ct_label": None}
        if r.planted and r.post_frames is not None:
            row["ct_label"] = pipeline.predict_window(r.post_frames, "post")["label"]
        rows.append(row)
    kept = [r for r in rows if r["confidence"] >= min_confidence]
    planted = [r for r in rows if r["ct_label"]]
    return {"rows": rows, "n_used": len(kept), "n_planted": len(planted),
            "tendencies": {c: sum(r["label"] == c for r in kept) / max(len(kept), 1) for c in T_CLASSES},
            "ct_tendencies": {c: sum(r["ct_label"] == c for r in planted) / max(len(planted), 1)
                              for c in CT_CLASSES},
            "agreement": sum(r["label"] == r["true"] for r in rows) / max(len(rows), 1),
            "ct_agreement": sum(r["ct_label"] == r["ct_true"] for r in planted) / max(len(planted), 1)}


report = scouting_report(pipeline, test_rounds[:40])
print(f"rounds analysed {report['n_used']}   attacker-tactic agreement {report['agreement']:.3f}")
for c, f in sorted(report["tendencies"].items(), key=lambda kv: -kv[1]):
    print(f"  {c:<9} {f*100:5.1f}%  {'#' * int(f * 40)}")
print(f"\nplanted rounds {report['n_planted']}   defender-response agreement {report['ct_agreement']:.3f}")
for c, f in sorted(report["ct_tendencies"].items(), key=lambda kv: -kv[1]):
    print(f"  {c:<9} {f*100:5.1f}%  {'#' * int(f * 40)}")


def plot_routes(rounds: List[Round], label: str, use_vision: bool = False):
    """Movement trails for one tactic — the view an analyst actually wants."""
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.imshow(plot_background(), extent=[0, 1, 1, 0])     # normalised coords, any image size
    for r in [x for x in rounds if x.label == label][:12]:
        pos = positions_from_video(detector, r.frames) if use_vision else r.positions
        for p in range(CFG.n_per_team):
            ax.plot(pos[:, p, 0], pos[:, p, 1], color="#ff9020", alpha=0.35, lw=1.2)
    ax.set_title(f"{label} — attacker routes ({'vision' if use_vision else 'oracle'})")
    ax.axis("off"); plt.tight_layout(); plt.show()


plot_routes(test_rounds, "A_SPLIT", use_vision=False)
plot_routes(test_rounds, "A_SPLIT", use_vision=True)

## 18. Objective 2 — where do they actually play?

Aggregate every tracked position into a 2-D histogram and lay it over the radar. This is the
cheapest objective in the project: the tracker already produces `(32, 10, 2)` per round, so a
tendency map is a `histogram2d` and a blur away. No extra model, no extra labels.

Three views, one subsection each:

* **18.1 Attacker vs defender occupancy** — where each side spends its time overall.
* **18.2 Per-tactic maps** — what an `A_SPLIT` looks like compared with an `A_RUSH`.
* **18.3 Route trails** — individual paths for every tactic, which read better than density
  for a small number of rounds.

A heatmap needs volume to mean anything. Under ~20 rounds you are looking at a handful of
paths smeared into a blob; prefer the trail view until you have more.

**Which rounds.** If `rounds_data.npz` exists (real labelled rounds from §16), positions come
from the vision pipeline — pixels → detector → tracker, exactly what deployment sees. Otherwise
the synthetic test rounds are used with their oracle positions.

In [ ]:
if Path(CFG.rounds_npz).exists():
    OBJ2_ROUNDS, OBJ2_VISION = load_real_rounds(), True    # real: pixels -> detector -> tracker
else:
    OBJ2_ROUNDS, OBJ2_VISION = test_rounds, False          # synthetic: oracle positions
print(f"Objective 2 on {len(OBJ2_ROUNDS)} rounds "
      f"({'vision' if OBJ2_VISION else 'oracle'} positions)")
print("attacker tactics:", {c: sum(r.label == c for r in OBJ2_ROUNDS) for c in T_CLASSES})
print("defender responses (planted rounds):",
      {c: sum(r.ct_label == c for r in OBJ2_ROUNDS) for c in CT_CLASSES})

_POS_CACHE: Dict[Tuple[int, str], np.ndarray] = {}


def round_positions(r: Round, use_vision: bool = False, detector=None,
                    phase: str = "pre") -> Optional[np.ndarray]:
    """(T, 10, 2) for one phase of one round. Vision positions are cached: one detector pass."""
    if not use_vision:
        return r.positions_of(phase)
    frames = r.frames_of(phase)
    if frames is None:
        return None
    key = (id(r), phase)
    if key not in _POS_CACHE:
        _POS_CACHE[key] = positions_from_video(detector, frames)
    return _POS_CACHE[key]


def team_slice(team: str) -> slice:
    return slice(0, CFG.n_per_team) if team == "T" else slice(CFG.n_per_team, N_PLAYERS)


def side_of(label: str) -> str:
    """Defender responses belong to CT; attacker tactics to T."""
    return "CT" if label in CT_CLASSES else "T"


def phase_of(label: str) -> str:
    """Attacker tactics are read before the plant, defender responses after it."""
    return "post" if label in CT_CLASSES else "pre"


def radar_bg() -> np.ndarray:
    return plot_background()


def occupancy_map(rounds: List[Round], team: str = "T", label: Optional[str] = None,
                  bins: int = 48, sigma: float = 1.2, use_vision: bool = False,
                  detector=None, phase: str = "pre") -> np.ndarray:
    """-> (bins, bins) normalised occupancy density for one side in one phase of the round."""
    sel = [r for r in rounds if (label is None or r.label_of(phase) == label)]
    sl = team_slice(team)
    xs, ys = [], []
    for r in sel:
        pos = round_positions(r, use_vision, detector, phase)
        if pos is None:
            continue
        p = pos[:, sl].reshape(-1, 2)
        p = p[np.isfinite(p).all(1)]
        xs.append(p[:, 0]); ys.append(p[:, 1])
    if not xs:
        return np.zeros((bins, bins), np.float32)
    H, _, _ = np.histogram2d(np.concatenate(ys), np.concatenate(xs), bins=bins,
                             range=[[0, 1], [0, 1]])
    if cv2 is not None and sigma > 0:
        H = cv2.GaussianBlur(H.astype(np.float32), (0, 0), sigma)
    return (H / H.max()).astype(np.float32) if H.max() > 0 else H.astype(np.float32)


def _heat_panel(ax, bg, H, team: str, title: str):
    ax.imshow(bg, extent=[0, 1, 1, 0])
    ax.imshow(H, extent=[0, 1, 1, 0], cmap="inferno" if team == "T" else "viridis",
              alpha=0.62, vmin=0, vmax=1, interpolation="bilinear")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

### 18.1 Attacker vs defender occupancy

Every round, both sides, side by side. The broadest question an analyst asks first: where does
each team spend its time?

In [ ]:
def show_side_occupancy(rounds: List[Round], use_vision: bool = False, detector=None,
                        bins: int = 48):
    """T (attackers) and CT (defenders) occupancy over all rounds."""
    bg = radar_bg()
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.0), layout="constrained")
    for ax, team in zip(axes, ("T", "CT")):
        H = occupancy_map(rounds, team, None, bins, use_vision=use_vision, detector=detector)
        _heat_panel(ax, bg, H, team, f"{'attackers (T)' if team == 'T' else 'defenders (CT)'}"
                                     f"\nn={len(rounds)} rounds")
    fig.suptitle(f"18.1 — side occupancy before the plant "
                 f"({'vision' if use_vision else 'oracle'} positions)")
    plt.show()


show_side_occupancy(OBJ2_ROUNDS, use_vision=OBJ2_VISION, detector=detector)

### 18.2 Per-tactic maps

Two grids. The first has one column per attacker tactic, drawn from the window before the
plant. The second has one column per defender response, drawn from the window after it. Each
grid has a row per side: the side running the tactic or response is marked with `*`; the other
row shows how the opponents were positioned. Panels are titled with their `n`; read anything
under ~20 as anecdote, not tendency.

In [ ]:
def show_tactic_occupancy(rounds: List[Round], labels: Sequence[str], phase: str,
                          use_vision: bool = False, detector=None, bins: int = 48):
    """2 x len(labels) grid: rows T / CT, columns labels. The side running it is marked with *."""
    bg = radar_bg()
    fig, axes = plt.subplots(2, len(labels), figsize=(3.0 * len(labels), 6.6), squeeze=False,
                             layout="constrained")
    for col, lab in enumerate(labels):
        n = sum(r.label_of(phase) == lab for r in rounds)
        for row, team in enumerate(("T", "CT")):
            H = occupancy_map(rounds, team, lab, bins, use_vision=use_vision, detector=detector,
                              phase=phase)
            star = " *" if team == side_of(lab) else ""
            _heat_panel(axes[row, col], bg, H, team,
                        f"{lab} — {team}{star}\nn={n}" + ("  (no rounds)" if n == 0 else ""))
    what = "attacker tactic, before the plant" if phase == "pre" else "defender response, after the plant"
    fig.suptitle(f"18.2 — occupancy per {what}; * = side running it "
                 f"({'vision' if use_vision else 'oracle'} positions)")
    plt.show()


show_tactic_occupancy(OBJ2_ROUNDS, T_CLASSES, "pre", use_vision=OBJ2_VISION, detector=detector)
show_tactic_occupancy(OBJ2_ROUNDS, CT_CLASSES, "post", use_vision=OBJ2_VISION, detector=detector)

### 18.3 Route trails — every tactic and every response

One figure for each attacker tactic (attackers' paths before the plant), then one for each
defender response (defenders' paths after the plant). Thin lines rather than density, so a
dozen rounds still reads as a dozen routes instead of a blob. The marker shows where each path
starts.

Identity is not promised (§2), so a line can jump between players when two teammates cross —
read the bundle of routes, not any single line.

In [ ]:
def _draw_trails(ax, bg, rounds: List[Round], label: str, team: str, max_rounds: int,
                 use_vision: bool, detector):
    phase = phase_of(label)
    sl = team_slice(team)
    colour = TEAM_EDGE[0] if team == "T" else TEAM_EDGE[1]
    sel = [r for r in rounds if r.label_of(phase) == label][:max_rounds]
    ax.imshow(bg, extent=[0, 1, 1, 0])
    for r in sel:
        pos = round_positions(r, use_vision, detector, phase)
        if pos is None:
            continue
        for pi in range(sl.start, sl.stop):
            p = pos[:, pi]
            p = p[np.isfinite(p).all(1)]                  # dead / never-seen slots
            if len(p) < 2:
                continue
            ax.plot(p[:, 0], p[:, 1], color=colour, alpha=0.35, lw=1.1)
            ax.plot(p[0, 0], p[0, 1], "o", color=colour, ms=2.5, alpha=0.6)
    total = sum(r.label_of(phase) == label for r in rounds)
    shown = f"n={len(sel)}" + (f" of {total}" if total > len(sel) else "")
    when = "before the plant" if phase == "pre" else "after the plant"
    ax.set_title(f"{label} — {'attackers' if team == 'T' else 'defenders'} {when}\n{shown}"
                 + ("  (no rounds)" if total == 0 else ""), fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


def show_trails(rounds: List[Round], label: str, team: Optional[str] = None,
                max_rounds: int = 15, use_vision: bool = False, detector=None):
    """One tactic or response, large. team=None -> the side running it."""
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    _draw_trails(ax, radar_bg(), rounds, label, team or side_of(label), max_rounds,
                 use_vision, detector)
    plt.tight_layout(); plt.show()


# one route-trail figure per attacker tactic, then one per defender response
for lab in T_CLASSES + CT_CLASSES:
    show_trails(OBJ2_ROUNDS, lab, use_vision=OBJ2_VISION, detector=detector)

## 19. Objective 3 — win rates, and an honest warning

**Win rate is a counting problem, not a vision problem.** The scoreboard already states the
result; inferring it from icons disappearing would be solving an easy problem the hard way and
getting it wrong — the radar cannot distinguish a bomb detonation from a defuse, and the round
timer sits outside your crop. So the winner is recorded as an **annotation** (one keypress in
`RoundLabeller`), not predicted by a model. That keeps the pixels-only rule intact, because
win/loss is a label, never a model input.

**And raw map win rate is not really your project's question.** "Does Mirage favour attack or
defence?" is answered by counting rounds, and public sites already track it. The version only
*your* system can answer is the conditional one:

> given that the attackers ran an **A_SPLIT**, how often did it work?
>
> once the bomb is down, does a **RETAKE** win more often than a **TRICKLE**?

Both need the classifier, which is what makes it worth building. Attacker tactics are scored as "did T win"; defender responses as "did CT win", over planted rounds only.

**Sample size is the binding constraint.** A win rate from 30 rounds carries a 95% interval of
roughly ±18 percentage points — wide enough that "42%" and "58%" are indistinguishable. The
table below prints Wilson intervals precisely so nobody quotes a number the data cannot
support. Plan your demo collection against it.

In [ ]:
def wilson_interval(wins: int, n: int, z: float = 1.96) -> Tuple[float, float, float]:
    """Wilson score interval — behaves properly at small n, unlike the normal approximation."""
    if n == 0:
        return (float("nan"), float("nan"), float("nan"))
    p = wins / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(centre - half, 0.0), min(centre + half, 1.0)


def win_rate_report(rounds: List[Round], min_n: int = 20) -> Dict:
    scored = [r for r in rounds if r.winner is not None]
    if not scored:
        print("no rounds carry a winner — label them in RoundLabeller (t / c keys)")
        return {}

    t_wins = sum(r.winner == "T" for r in scored)
    p, lo, hi = wilson_interval(t_wins, len(scored))
    print(f"OVERALL  ({len(scored)} rounds)")
    print(f"  T side wins   {p*100:5.1f}%   95% CI [{lo*100:.1f}, {hi*100:.1f}]")
    print(f"  CT side wins  {(1-p)*100:5.1f}%   95% CI [{(1-hi)*100:.1f}, {(1-lo)*100:.1f}]")
    verdict = ("too few rounds to call" if len(scored) < min_n else
               "T-sided" if lo > 0.5 else "CT-sided" if hi < 0.5 else
               "no significant side advantage — the interval spans 50%")
    print(f"  verdict: {verdict}\n")

    out = {"overall": {"n": len(scored), "t_win_rate": p, "ci": (lo, hi), "verdict": verdict},
           "by_tactic": {}, "by_response": {}}
    groups = (("by_tactic", "BY ATTACKER TACTIC - did the attackers win the round?",
               T_CLASSES, lambda r: r.label, lambda r: r.tactic_worked),
              ("by_response", "BY DEFENDER RESPONSE - after a plant, did the defenders win?",
               CT_CLASSES, lambda r: r.ct_label, lambda r: r.response_worked))
    for key, head, labels, get, worked in groups:
        print(head)
        print(f"  {'label':<10}{'n':>5}{'success':>10}{'95% CI':>18}")
        for c in labels:
            sub = [r for r in scored if get(r) == c]
            if not sub:
                continue
            w = sum(bool(worked(r)) for r in sub)
            pp, l, h = wilson_interval(w, len(sub))
            flag = "" if len(sub) >= min_n else "  <- too few to quote"
            print(f"  {c:<10}{len(sub):>5}{pp*100:>9.1f}%   [{l*100:5.1f}, {h*100:5.1f}]   "
                  f"{'#' * int(pp * 22)}{flag}")
            out[key][c] = {"n": len(sub), "success_rate": pp, "ci": (l, h)}
        print()
    return out


def plot_win_rates(report: Dict, min_n: int = 20):
    panels = [("by_tactic", "attacker tactics: did T win?", "#d9822b"),
              ("by_response", "defender responses after a plant: did CT win?", "#2f7fd0")]
    panels = [p for p in panels if report.get(p[0])]
    if not panels:
        return
    fig, axes = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 3.6), squeeze=False,
                             gridspec_kw={"width_ratios": [len(report[p[0]]) for p in panels]})
    for ax, (key, title, colour) in zip(axes[0], panels):
        bt = report[key]
        names = list(bt)
        vals = [bt[c]["success_rate"] for c in names]
        los = [vals[i] - bt[c]["ci"][0] for i, c in enumerate(names)]
        his = [bt[c]["ci"][1] - vals[i] for i, c in enumerate(names)]
        cols = [colour if bt[c]["n"] >= min_n else "#c8c8c8" for c in names]
        ax.bar(names, vals, yerr=[los, his], capsize=5, color=cols)
        ax.axhline(0.5, ls="--", c="grey", lw=1)
        ax.set_ylim(0, 1); ax.set_title(title, fontsize=10)
        for k, c in enumerate(names):
            ax.text(k, 0.04, f"n={bt[c]['n']}", ha="center", fontsize=8, color="white")
        ax.tick_params(axis="x", labelrotation=20)
    axes[0][0].set_ylabel("success rate")
    fig.suptitle("Objective 3 — success with 95% Wilson intervals (grey = too few rounds to quote)",
                 fontsize=10)
    plt.tight_layout(); plt.show()


def rounds_needed(target_halfwidth: float = 0.10, p: float = 0.5, z: float = 1.96) -> int:
    return int(math.ceil(z * z * p * (1 - p) / (target_halfwidth ** 2)))


WR = win_rate_report(train_rounds + val_rounds + test_rounds)
plot_win_rates(WR)
print("\nrounds needed per group for a given precision (at p=0.5):")
for hw in (0.20, 0.15, 0.10, 0.05):
    print(f"  +/- {hw*100:4.0f} pp  ->  {rounds_needed(hw):>4} rounds")
need = rounds_needed(0.10)
plant_rate = np.mean([r.planted for r in train_rounds + val_rounds + test_rounds])
print(f"\nAt +/-10pp each group needs ~{need} rounds. Attacker tactics: {len(T_CLASSES)} x {need} = "
      f"{need * len(T_CLASSES)} rounds.\nDefender responses: {len(CT_CLASSES)} x {need} = "
      f"{need * len(CT_CLASSES)} planted rounds, which come from the same rounds "
      f"(~{int(need * len(CT_CLASSES) / plant_rate)} at a {plant_rate:.0%} plant rate).\n"
      f"So ~{need * len(T_CLASSES)} labelled rounds (~{need * len(T_CLASSES) // 25} matches) answers both. "
      "Plan collection against this, not against a guess.")

## 20. Saving

In [ ]:
out = Path(CFG.out_dir)
torch.save({"detector": detector.state_dict(),
            "heads": {ph: {"classifier": RESULTS[ph][BEST[ph]]["model"].state_dict(),
                           "classifier_name": BEST[ph], "classes": PHASES[ph],
                           "normaliser": [t.cpu() for t in DATA[ph]["train"].normaliser]}
                      for ph in ("pre", "post")},
            "repr_mode": REPR, "crop_box": CROP, "config": asdict(CFG)}, out / "stratscout.pt")

as_lists = lambda d: {k: (list(v) if isinstance(v, tuple) else v) for k, v in d.items()}
summary = {
    "mode": "real" if USING_REAL_ICONS else "synthetic",
    "crop_box": list(CROP),
    "detector": {k: float(v) for k, v in DET_METRICS.items()},
    "pipeline_pos_err_px": float(np.mean(errs) * CFG.minimap_px),
    "classifiers": {ph: {n: {"val_macro_f1": r["val"]["macro_f1"],
                             "test_macro_f1": r["test"]["macro_f1"],
                             "test_accuracy": r["test"]["accuracy"]} for n, r in RESULTS[ph].items()}
                    for ph in RESULTS},
    "experiment_A_oracle": {"attacker": {"accuracy": exp_a["accuracy"], "macro_f1": exp_a["macro_f1"]},
                            "defender": {"accuracy": ct_a["accuracy"], "macro_f1": ct_a["macro_f1"]}},
    "experiment_B_vision": {"attacker": {"accuracy": exp_b["accuracy"], "macro_f1": exp_b["macro_f1"]},
                            "defender": {"accuracy": ct_b["accuracy"], "macro_f1": ct_b["macro_f1"]}},
    "experiment_Bprime_vision_trained": (
        {"accuracy": exp_bp["accuracy"], "macro_f1": exp_bp["macro_f1"]} if exp_bp else None),
    "objective3_win_rates": {
        "overall": as_lists(WR.get("overall", {})),
        "by_tactic": {c: as_lists(d) for c, d in WR.get("by_tactic", {}).items()},
        "by_response": {c: as_lists(d) for c, d in WR.get("by_response", {}).items()}},
}
(out / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 21. What to replace next

Ordered by risk removed, not by how interesting it is.

| # | Task | Replaces |
|---|---|---|
| 1 | Record demo renders with a locked radar; confirm Section 3.1 reports a low difference | nothing — this gates everything |
| 2 | `auto_label_icons` + `ReviewTool` on ~200 frames | synthetic icon labels |
| 3 | `RoundLabeller` across 8–10 matches (~200 rounds) | `generate_split` |
| 4 | Retrain both models on real rounds; re-run Experiment A vs B | synthetic numbers |
| 5 | Decide the deployment window strategy (Section 16) | sliding-window default |

**Three things this prototype deliberately does not do**, so you notice their absence:

* **No dead-player handling in the tracker.** Real rounds lose icons permanently. The tracker
  coasts and then sits still, which quietly drags the centroid feature. Fix this early — it is
  a correctness bug, not a refinement.
* **No class-imbalance handling.** `DEFAULT` will dominate real rounds and `SAVE` will be rare.
  Add class weights to the cross-entropy once you see real counts.
* **No plant detection from pixels.** The labeller marks the plant by hand. At deployment the
  planted bomb has to be spotted on the radar, or the after-plant question cannot be asked.
* **No annotator agreement check.** Have two people label the same 30 rounds. If humans agree
  only 70% of the time on `A_RUSH` versus `A_SPLIT`, then ~70% is your model's ceiling, and
  knowing that number early tells you when to stop tuning.

**The class scheme, settled.** The old single list had no bucket for slow default rounds and
mixed one defender class into an attacker head. Now each round answers two questions: the
attackers' tactic before the plant (with `DEFAULT` for map control), and the defenders'
response after it (`RETAKE`, `TRICKLE`, `SAVE`). The site is read from the plant, not labelled.